# M53 · Moment-tensor inversion: least squares, misfit norms, time shifts and MTUQ

**Week 7 · about 11 h** · before this: M20, M41, M50, M51, M52

> **한국어 요약** 이 모듈에서는 M52와 반대 방향으로, 기록과 그린 함수에서 지진의 모멘트 텐서를 거꾸로 찾습니다. 먼저 정답을 아는 가상의 관측망에서 잡음과 빠진 관측소가 있는 자료로 선형 최소제곱 역산을 직접 작성하고, 여러 번의 잡음 실현으로 추정값의 공분산이 맞는지 확인합니다. 한 관측소가 7배 크게 기록된 동료의 L2 격자 탐색을 먼저 예측한 뒤, L2·L1·혼합 노름과 참여 비율(유효 관측 수)로 어느 관측소가 답을 쓰는지 재고, 시간 이동의 한계를 넓힐 때 반대 메커니즘의 맞춤이 얼마나 좋아지는지 측정합니다. 모멘트 텐서를 룬 좌표와 이중 짝힘 단층면으로 바꾸어 M51의 카간 각으로 비교합니다. 마지막으로 2019년 상주 지진에서 직접 고른 위상으로 시간창을 정하고, PyFK로 MTUQ가 읽는 그린 함수 데이터베이스를 만들어 진폭 점검을 먼저 한 뒤 최소제곱 역산과 MTUQ 이중 짝힘 격자 탐색(주향·경사·미끄럼각·규모·깊이)을 실행하고, KIGAM 단층면해와 M51의 SKHASH 결과와 비교하며, 그대로 다시 실행한 뒤 노름 하나를 바꾸어 보고(MTUQ가 이름 그대로 계산하는 L2와 혼합 노름 가운데서), 정해진 기준으로 '결정됨', '잠정', '결정되지 않음'을 스스로 판정합니다.

**You will**

- Write a weighted linear least-squares moment-tensor inversion that handles data gaps and a deviatoric constraint, and show over many noise draws that its covariance describes its errors
- Measure with the L2, L1 and hybrid norms and the participation ratio which traces write the answer of an inversion, including one with a station recorded seven times too loud
- Find the best bounded time shift between a record and a synthetic, and measure how much a looser bound lets the opposite mechanism fit
- Turn a moment tensor into its lune coordinates and its double-couple planes and compare mechanisms with the Kagan angle
- Run MTUQ for the 2019 Sangju mainshock with windows from your own picks and Green's functions from PyFK after an amplitude check, compare it with KIGAM, SKHASH and your least squares, rerun it unchanged, change the norm, and decide by stated rules whether the mechanism is resolved

**New words:** waveform inversion (파형 역산), moment tensor (모멘트 텐서), double couple (이중 우력, 더블 커플), CLVD (보상 선형 벡터 쌍극자), isotropic component (등방 성분), deviatoric moment tensor (편차 모멘트 텐서), up-south-east coordinates (USE; 위-남-동 좌표계), north-east-down coordinates (NED; 북-동-아래 좌표계) · [all terms](../../../docs/glossary.md)

**Optional reading:** Aki & Richards (2002), chapter 4 (the moment tensor and the representation theorem) and Box 4.4; Shearer (2019), the chapter on earthquakes (moment tensors and their inversion); Zhu & Helmberger (1996), the whole paper (the cut-and-paste method: body and surface windows, time shifts, the L1 and L2 norms); Tape & Tape (2012), sections 1-3 (the lune and the eigenvalues of a moment tensor); Thurin et al. (2025), the introduction and the description of the misfit function and its time shifts

Log any AI help in `work/ai_log/M53.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M53.yaml](../../../.sgtl/criteria/M53.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
BAND = "HH"                               # broadband channels
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks of module M20 (in work/picks/)
SKHASH_CSV = "../M51_first_motions/results/M51_skhash_first/mechanisms.csv"   # your SKHASH run of task M51-T5
KIGAM_PLANE_A = (288.0, 89.0, 7.0)        # strike, dip, rake (degrees); KIGAM (2021), report KEMS-2019-062
KIGAM_PLANE_B = (198.0, 83.0, 179.0)      # the other nodal plane of the same report
KIGAM_MW = 3.8                            # moment magnitude of the same report
KIGAM_DEPTH_KM = 12.8                     # focal depth of the same report (km)
VELOCITY_MODEL = "kim1983"                # layered model of the Green's functions (Kim and Kim 1983)
DEPTHS_KM = [6, 8, 10, 12, 14, 16, 18]    # Part 5-6: trial depths (whole km; never 15, a boundary of kim1983)
REFERENCE_DEPTH_KM = 14                   # Part 5: depth of the origin-time correction (the KMA depth)
MAGNITUDES = [3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 4.0, 4.1]   # Part 6: trial moment magnitudes
FK_DT_S, FK_NPT = 0.1, 1024               # PyFK samples of the Green's functions: interval (s) and number (102.4 s)
STF_DURATION_S = 0.4                      # duration of PyFK's trapezoid source time function (s), as in M52
PRE_FILT_HZ = (0.02, 0.04, 8.0, 10.0)     # response removal (M10): frequency taper (Hz)
DECIMATE = 10                             # records: 100 samples per second -> 10 (the samples of FK_DT_S)
BAND_HZ = (0.08, 0.30)                    # band-pass of records and Green's functions (Hz; causal, 4 poles)
WINDOW_S = 40.0                           # length of the window that MTUQ cuts around your S pick (s)
MAX_SHIFT_S = 1.0                         # largest time shift allowed between record and synthetic (s)
NORM = "hybrid"                           # Part 6: misfit norm of the first MTUQ run ("L2" or "hybrid", Part 2)
CHANGED_NORM = "L2"                       # Part 6: the one parameter you change in the third run ("L2" or "hybrid")
QC_MAX_DEV = 0.6                          # Part 5: largest accepted |deviation| of log10(peak x distance) (0.6: x4)
GRID_POINTS = 20                          # Part 6: points per axis of MTUQ's double-couple grid
ACCEPT_TOL = 0.02                         # Part 6: acceptable mechanisms lie within 2 % of the best misfit
RULES = {"min_stations": 4, "min_n_eff": 3.0, "max_share": 0.5,               # below these: not resolved
         "good_n_eff": 6.0, "good_share": 0.3, "max_spread_deg": 30.0, "max_kagan_lsq_deg": 30.0}
TOY_SEED = 53                             # Parts 1-4: the made-up network
TOY_MECHANISM = (35.0, 60.0, -40.0)       # Parts 1-4: the made-up fault (strike, dip, rake in degrees)
TOY_MW = 3.5                              # Parts 1-4: its moment magnitude
TOY_LOUD = "SY.D04"                       # Part 2: the made-up station whose records are too loud
TOY_GAIN = 7.0                            # Part 2: by this factor
TOY_DELAY_S = 0.8                         # Part 3: travel-time errors of the made-up stations, up to +-0.8 s
TOY_MAX_SHIFT_S = 1.0                     # Part 3: the bound of your shifts on the made-up network (s)
TOY_BOUNDS_S = [0.5, 1.0, 2.0, 3.0, 4.0]  # Part 3: bounds tried against the opposite mechanism (s)
TOY_BAND_HZ = (0.10, 0.25)                # Part 3: the band of that test (Hz)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files (database, runs, tables) go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M53")

import hashlib
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display


def show_figure(figure, filename):
    """Save and embed the same scientific plot in the notebook."""
    output = Path(FIGURE_DIR) / filename
    figure.savefig(output, dpi=150, bbox_inches="tight")
    display(Image(filename=str(output)))
    plt.close(figure)

from obspy import Trace, UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth
from sgtl_tutorial import pyfk_runner as fk
from sgtl_tutorial._paths import short_path

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M53-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up networks, records and tensors whose answers
are known from how they were built (`st.reroll("M53-T1")` draws new ones); tasks 5 and 6 also recompute what you
wrote about the Sangju earthquake from your own files and never print a value of their own for them.

The module climbs a ladder in three steps. **R1**: write the core functions (a least-squares inversion, a misfit
budget, a bounded time shift, the lune and the planes of a tensor) and test them on a made-up network with a
known answer. **R2**: apply them to the records of the 2019 Sangju earthquake and compare with an independent
mechanism (KIGAM's). **R3**: run the production tool, MTUQ: once, again unchanged, and once with one parameter
changed, and decide by stated rules what the result is worth.

This module needs your STP download `sangju2019` (module M03), your manual picks of module M20, your SKHASH run of
module M51 (task 5), the PyFK worker environment (`bin/sgtl-install pyfk --apply`, module M52) and MTUQ
(`bin/sgtl-install mt --apply`; inspect the dry run first). MTUQ runs here on one core, without MPI (a system
for coordinating several worker processes). Keep the **SGTL (Python 3.12)** kernel; the PyFK calls start their
own worker environment. Parts 1-4 need none of the downloads or production tools, but do need the earlier
functions requested in Part 0.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M53.md`.

## Part 0 · From records back to the source (about 30 min)

Module M52 went forward: a moment tensor and a model Earth gave **synthetic seismograms**. A **waveform
inversion** (glossary `waveform_inversion`) goes back: from records and **Green's functions** (glossary
`greens_function`) to the moment tensor. For a point source the record is linear in the six independent elements
of the **moment tensor** (glossary `moment_tensor`). Write them as a column $\mathbf{m}$ in the **up-south-east**
frame (glossary `up_south_east`) that MTUQ uses, $\mathbf{m} = (M_{rr}, M_{\theta\theta}, M_{\phi\phi},
M_{r\theta}, M_{r\phi}, M_{\theta\phi})$ in N m. Every sample of every record is then a sum

$$d_i = \sum_{j=1}^{6} G_{ij}\, m_j ,\qquad \mathbf{d} = \mathbf{G}\,\mathbf{m},$$

where column $j$ of $\mathbf{G}$ is the **elementary seismogram** (glossary `elementary_seismogram`): the records
that a source with $m_j = 1$ N m and every other element zero would make, all stations and components one after
another. Finding $\mathbf{m}$ from $\mathbf{d}$ is then a linear problem (Aki & Richards 2002, chapter 4; Jost &
Herrmann 1989; Minson & Dreger 2008). Three things make it hard in practice, and each part of this notebook takes
one: noise and missing records (Part 1), records whose size is wrong (Part 2), and a model Earth whose travel times
are a little off (Part 3).

A tensor is more than its six numbers. Its three **eigenvalues** (glossary `eigenvalue`) say what kind of source
it is: $(M_0, 0, -M_0)$ for a **double couple** (glossary `double_couple`), three equal values for a pure
**isotropic** source (glossary `isotropic_component`, a change of volume), $(2, -1, -1)$ or $(1, 1, -2)$ times a
constant for a **CLVD** (glossary `clvd`). A tensor whose eigenvalues add up to zero is **deviatoric** (glossary
`deviatoric_moment_tensor`): no change of volume. Its eigenvectors are the T, P and B axes of module M50.

This module uses three functions that you wrote earlier. Paste your `moment_tensor` and your `magnitude_moment` of
module M52 (tasks M52-T1 and M52-T2) and your `kagan_angle` of module M51 (task M51-T3) into the next cell, as they
are in your notebooks (they need numpy only), and run it.

In [ ]:
# Paste your moment_tensor and magnitude_moment of module M52 and your kagan_angle of module M51 below this line.

In [ ]:
for name, task in (("moment_tensor", "M52-T1"), ("magnitude_moment", "M52-T2"), ("kagan_angle", "M51-T3")):
    if not callable(globals().get(name)):
        raise NameError(f"{name} not found: paste your function into the cell above and run it (finish task "
                        f"{task} of module {task[:3]} first).")


def six(tensor):
    """The six independent elements of a symmetric 3 x 3 tensor, in the order (11, 22, 33, 12, 13, 23)."""
    t = np.asarray(tensor, dtype=float)
    return np.array([t[0, 0], t[1, 1], t[2, 2], t[0, 1], t[0, 2], t[1, 2]])


def full(m6):
    """The symmetric 3 x 3 tensor of six elements in the order (11, 22, 33, 12, 13, 23)."""
    a = np.asarray(m6, dtype=float)
    return np.array([[a[0], a[3], a[4]], [a[3], a[1], a[5]], [a[4], a[5], a[2]]])


TOY_M0 = magnitude_moment(mw=TOY_MW)
TOY_TRUE = six(moment_tensor(*TOY_MECHANISM, TOY_M0, frame="USE"))
print("made-up fault", TOY_MECHANISM, f"Mw {TOY_MW}: m (USE, N m) =", np.array2string(TOY_TRUE, precision=3))
print("eigenvalues / M0:", np.round(np.linalg.eigvalsh(full(TOY_TRUE)) / TOY_M0, 6))

**The made-up network.** Parts 1-4 practise on eight made-up stations with invented codes (`SY.D01` ... `SY.D08`).
Their elementary seismograms are not physics: each is a sum of two smooth pulses at a P-like and an S-like time,
with random sizes and signs, 200 samples at 0.1 s per component (Z, R, T), in metres of displacement per N m. That
is enough to practise the linear algebra; Parts 5 and 6 use real Green's functions. `toy_network` makes them, and
`toy_records` the records of a tensor with noise.

In [ ]:
TOY_DT_S = 0.1
TOY_CODES = [f"SY.D{k:02d}" for k in range(1, 9)]


def toy_network(seed, n_stations=8, n_samples=200, dt_s=TOY_DT_S):
    """Made-up elementary seismograms: array (stations, 3 components Z R T, samples, 6 elements in USE order)."""
    gen = np.random.default_rng(seed)
    t = np.arange(n_samples) * dt_s
    kernels = np.zeros((n_stations, 3, n_samples, 6))
    distance_km = gen.uniform(15.0, 60.0, n_stations)
    for s in range(n_stations):
        p_pulse = np.gradient(np.exp(-((t - distance_km[s] / 6.0 - 1.0) / 1.0) ** 2), dt_s)
        s_pulse = np.gradient(np.exp(-((t - distance_km[s] / 3.5 - 1.0) / 1.4) ** 2), dt_s)
        for c in range(3):
            p_size = gen.normal(size=6) * (0.0 if c == 2 else 0.6)
            s_size = gen.normal(size=6)
            kernels[s, c] = (np.outer(p_pulse, p_size) + np.outer(s_pulse, s_size)) * 1e-16 / (distance_km[s] / 30.0)
    return kernels


def toy_records(kernels, m6, noise_fraction, seed):
    """Records (stations, 3, samples) of the tensor m6 with Gaussian noise of noise_fraction x the largest sample."""
    clean = np.einsum("scnj,j->scn", kernels, np.asarray(m6, dtype=float))
    sigma = noise_fraction * np.abs(clean).max()
    return clean + np.random.default_rng(seed).normal(0.0, sigma, clean.shape), sigma


TOY_G = toy_network(TOY_SEED)
toy_obs, toy_sigma = toy_records(TOY_G, TOY_TRUE, 0.05, TOY_SEED + 1)
print("made-up network:", TOY_G.shape, "(stations, components, samples, elements); noise sigma",
      f"{toy_sigma:.3e} m")

## Part 1 · Linear least squares, with noise and gaps (about 70 min, then task 1)

Stack every record into one column $\mathbf{d}$ ($n$ samples) and the elementary seismograms into $\mathbf{G}$
($n \times 6$). Each sample has noise of standard deviation $\sigma_i$. **Least squares** (glossary `least_squares`)
with weights chooses the $\mathbf{m}$ that makes $\sum_i (d_i - (\mathbf{G}\mathbf{m})_i)^2/\sigma_i^2$ smallest.
With $\mathbf{W} = \mathrm{diag}(1/\sigma_i^2)$ the answer solves the **normal equations** (glossary
`normal_equations`)

$$(\mathbf{G}^{\mathsf T}\mathbf{W}\mathbf{G})\,\hat{\mathbf{m}} = \mathbf{G}^{\mathsf T}\mathbf{W}\mathbf{d},
\qquad \mathbf{C}_m = (\mathbf{G}^{\mathsf T}\mathbf{W}\mathbf{G})^{-1}.$$

$\mathbf{C}_m$ is the **covariance matrix** (glossary `covariance_matrix`) of $\hat{\mathbf{m}}$: if the noise is
Gaussian with those $\sigma_i$, the error $\hat{\mathbf{m}} - \mathbf{m}_{\mathrm{true}}$ scatters with that
covariance. One number tests it: the **chi-square statistic** (glossary `chi_square_statistic`)
$\chi^2 = \mathbf{e}^{\mathsf T}\mathbf{C}_m^{-1}\mathbf{e}$ of the error $\mathbf{e}$ averages the number of free
parameters over many noise draws (6 here).

Two complications. A **data gap** (glossary `data_gap`): a station that recorded nothing, or a dead component,
leaves samples marked NaN (not a number); they must leave the sums, not enter them as zeros. A **deviatoric**
constraint: Green's functions computed for shear sources only (PyFK's double-couple sources, Part 5) cannot see a
change of volume, so $\mathbf{G}$ maps the isotropic tensor $(1, 1, 1, 0, 0, 0)$ to zero and the normal matrix is
singular. Then you look for $\mathbf{m}$ among the tensors with zero trace only: write $\mathbf{m} =
\mathbf{B}\mathbf{p}$, with the five columns of $\mathbf{B}$ trace-free tensors that span all of them, solve for
the five numbers $\mathbf{p}$, and turn the answer and its covariance back with $\mathbf{B}$.

### M53-T1 · Weighted least squares for a moment tensor · core · about 75 min

**Goal:** write `invert_mt(G, d, sigma, deviatoric=False)`: the weighted least-squares moment tensor and its
covariance.

**Given:** Part 1 and numpy (`numpy.linalg.solve` and `numpy.linalg.inv` are fine). Not allowed: ready-made
least-squares and pseudo-inverse routines (`numpy.linalg.lstsq`, `numpy.linalg.pinv`, SciPy, scikit-learn,
statsmodels), pyrocko, MTUQ and the course package (the check compares with one of them).

**Contract:**
- Inputs: `G`, an array of shape (n, 6): the elementary seismograms, columns in the order (Mrr, Mtt, Mpp, Mrt, Mrp,
  Mtp), in data units per N m; `d`, an array of n samples, NaN where the record has a gap; `sigma`, the noise
  standard deviation of each sample (one number for all, or an array of n positive numbers); `deviatoric`: when
  True, the answer must have Mrr + Mtt + Mpp = 0 exactly (the best fit among trace-free tensors). Do not change
  the inputs.
- Skip every sample whose `d` is NaN; use every other sample with the weight 1 / sigma².
- Output: a tuple `(m, cov)`: `m`, an array of 6 numbers (N m, the same order), and `cov`, its 6 x 6 covariance
  (N² m²). With `deviatoric=True`, `cov` is the covariance of the constrained answer (its rank is 5).
- Example (invented): `G` = the 6 x 6 identity stacked twice (12 samples), `d` = two copies of (1, 2, 3, 4, 5, 6),
  `sigma` = 1 give `m` = (1, 2, 3, 4, 5, 6) and `cov` = 0.5 times the identity.

**Done when** `st.check("M53-T1", func=invert_mt)` passes: made-up networks without noise (with and without gaps,
with unequal sigma, both settings of `deviatoric`, also one whose kernels cannot see a change of volume), compared
with an independent least-squares routine; then 300 noise draws of one network, where your covariance must
describe the scatter of your errors (the mean chi-square and the correlations of the errors).

**Stuck?** `st.hint("M53-T1", 1)`

In [ ]:
# Before you code: half of the stations of a network go silent. Will the least-squares tensor be biased, or only less
# certain, and how should the covariance change? If every sigma doubles but the data stay the same, what happens to
# m and to cov? What goes wrong in the normal equations when the kernels cannot see a change of volume?
st.predict("M53-T1", """

""")

In [ ]:
def invert_mt(G, d, sigma, deviatoric=False):
    """Weighted least-squares moment tensor (USE order, N m) and its covariance.

    G          : (n, 6) elementary seismograms, columns Mrr, Mtt, Mpp, Mrt, Mrp, Mtp (data units per N m)
    d          : (n,) data; NaN marks a gap
    sigma      : noise standard deviation per sample (number or (n,) array)
    deviatoric : True -> best trace-free tensor
    returns    : (m, cov): (6,) array in N m and (6, 6) covariance in N^2 m^2
    """
    # Step 1: keep the rows where d is not NaN; the weight of each kept row is 1 / sigma^2.
    # Step 2: the basis B: the 6 x 6 identity, or for deviatoric 5 trace-free columns (two for the diagonal, three
    #         off-diagonal unit columns).
    # Step 3: with A = (G B)^T W (G B) and b = (G B)^T W d, solve A p = b; m = B p and cov = B A^-1 B^T.
    # YOUR CODE HERE
    raise NotImplementedError("M53-T1: write your code here")

In [ ]:
st.check("M53-T1", func=invert_mt)

Your function on the made-up network: the records of the made-up fault with 5 % noise, then with stations
`SY.D02` and `SY.D06` silent (gaps), then the deviatoric version. For each, the tensor in units of the true $M_0$,
the largest error, and the error measured in your own standard deviations (the square root of the diagonal of
`cov`).

In [ ]:
def stack(records, kernels):
    """Records (stations, 3, n) and kernels (stations, 3, n, 6) as one column d and one matrix G."""
    return records.reshape(-1), kernels.reshape(-1, 6)


toy_d, toy_Gmat = stack(toy_obs, TOY_G)
gappy = toy_obs.copy()
gappy[[1, 5]] = np.nan
rows = []
for label, data, dev in (("all stations", toy_d, False), ("two silent", gappy.reshape(-1), False),
                         ("two silent, deviatoric", gappy.reshape(-1), True)):
    m_hat, cov_hat = invert_mt(toy_Gmat, data, toy_sigma, deviatoric=dev)
    err = m_hat - TOY_TRUE
    rows.append({"case": label, "m / M0": np.round(m_hat / TOY_M0, 3).tolist(),
                 "largest |error| / M0": round(float(np.abs(err).max() / TOY_M0), 4),
                 "largest |error| / own sd": round(float(np.max(np.abs(err) / np.sqrt(np.diag(cov_hat)))), 2),
                 "trace / M0": round(float(m_hat[:3].sum() / TOY_M0), 5)})
print("true m / M0:", np.round(TOY_TRUE / TOY_M0, 3).tolist())
print(pd.DataFrame(rows).to_string(index=False))

**See which tensor components the data constrain.** The stars are the known synthetic source; the other
symbols are your inversions with one-standard-deviation bars from your covariance matrix. Dividing by the
same true scalar moment keeps all six components comparable. Missing stations change the information in
the fit; imposing zero trace constrains a different model, rather than recovering missing observations.

**Interpret:** which components become less certain when stations are silent? Do the true values lie within
every one-standard-deviation bar, and should you expect that in one noise realization?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
visual_x = np.arange(6)
ax.plot(visual_x, TOY_TRUE / TOY_M0, "k*", ms=12, label="Known source")
for offset, (label, data, dev) in zip((-0.18, 0, 0.18),
        (("All stations", toy_d, False), ("Two silent", gappy.reshape(-1), False),
         ("Two silent, zero trace", gappy.reshape(-1), True))):
    visual_m, visual_cov = invert_mt(toy_Gmat, data, toy_sigma, deviatoric=dev)
    ax.errorbar(visual_x + offset, visual_m / TOY_M0, yerr=np.sqrt(np.diag(visual_cov)) / TOY_M0,
                fmt="o", ms=4, capsize=3, label=label)
ax.set(xticks=visual_x, xticklabels=["rr", "θθ", "φφ", "rθ", "rφ", "θφ"], xlabel="Moment-tensor component (USE)",
       ylabel="Component / true scalar moment", title="Your recovered tensor and formal uncertainty")
ax.axhline(0, color="0.6", lw=0.7)
ax.legend(fontsize=8, ncol=2)
show_figure(fig, "M53_tensor_recovery.png")

**Explain (3-5 sentences):** Use the printed rows. How large are the errors in units of your own standard
deviations, and what does that say about your covariance? What did silencing two stations do to the errors and to
the standard deviations? What did the deviatoric constraint change, and why is the trace of the made-up fault
zero anyway? Was your prediction right?

## Part 2 · Which traces write the answer? (about 60 min, then task 2)

The least-squares fit of Part 1 is the **L2 norm** of the residuals: their squares added up. A **misfit function**
(glossary `misfit_function`) can add them up in other ways (glossary `l1_l2_norm`; Zhu & Helmberger 1996). MTUQ
(glossary `mtuq`; Thurin et al. 2025) defines three, for a set of traces $k$ (one station, one component each)
with residuals $r_k(t_i) = d_k(t_i) - s_k(t_i)$ at the sample interval $\Delta t$:

$$\text{L2:}\;\; \sum_k \Delta t\sum_i r_k(t_i)^2,\qquad \text{L1:}\;\; \sum_k \Delta t\sum_i |r_k(t_i)|,\qquad
\text{hybrid:}\;\; \sum_k \Delta t\,\sqrt{\textstyle\sum_i r_k(t_i)^2}.$$

The **hybrid norm** (glossary `hybrid_norm`) is L2 inside each trace and L1 across traces. Each trace adds one term
to the sum; its **share** is that term divided by the total. If one trace has all of the misfit, the answer is
written by that trace alone. The **participation ratio** (glossary `participation_ratio`)

$$n_{\mathrm{eff}} = \frac{1}{\sum_k \mathrm{share}_k^2}$$

counts how many traces effectively decide: it is $K$ when $K$ traces share the misfit equally and 1 when one trace
has it all. A misfit total and a good-looking fit say nothing about this; the shares do.

A note for Part 6. In the MTUQ version of this course (0.2.0), the fast code of MTUQ's grid search computes the L2
sum when it is asked for `"L1"`: in MTUQ's source, `mtuq/misfit/waveform/level2.py` sends the names `"L2"` and
`"L1"` to the same L2 routine. A run named L1 would be an L2 run under the wrong name, so Part 6 runs MTUQ only
with `"L2"` and `"hybrid"`, the two norms it computes as named. Your `misfit_budget` computes all three.

A colleague ran a grid search on the made-up network over strike, dip and rake (15-degree steps) and five
magnitudes, with the L2 norm. The colleague did not know that the records of station `TOY_LOUD` came with a gain
`TOY_GAIN` times too large (a wrong number in that station's response). `toy_search` is the grid search: it builds
each trial tensor with your `moment_tensor`, its synthetics from the kernels, and asks `misfit(obs, syn, dt_s,
norm)` for a dictionary whose `"total"` it minimizes.

In [ ]:
TOY_GRID = [(s, d, r) for s in range(0, 360, 15) for d in range(15, 91, 15) for r in range(-165, 181, 15)]
TOY_MWS = [3.3, 3.4, 3.5, 3.6, 3.7]
TOY_UNIT = np.array([six(moment_tensor(s, d, r, 1.0, frame="USE")) for s, d, r in TOY_GRID])


def toy_search(records, kernels, norm, misfit, keep=None):
    """Grid search on the made-up network: (best (strike, dip, rake), its Mw, its synthetics, its misfit total)."""
    keep = list(range(len(kernels))) if keep is None else list(keep)
    obs = records[keep].reshape(-1, records.shape[-1])
    best = (np.inf, None, None, None)
    for mw in TOY_MWS:
        trial = np.einsum("scnj,kj->kscn", kernels[keep], TOY_UNIT * magnitude_moment(mw=mw))
        trial = trial.reshape(len(TOY_UNIT), -1, records.shape[-1])
        for k in range(len(TOY_UNIT)):
            total = misfit(obs, trial[k], TOY_DT_S, norm)["total"]
            if total < best[0]:
                best = (total, TOY_GRID[k], mw, trial[k])
    return best[1], best[2], best[3], best[0]

### M53-T2 · The misfit budget of a fit · core · about 60 min

**Goal:** write `misfit_budget(obs, syn, dt_s, norm="L2")`: each trace's term of the misfit, the total, the shares
and the participation ratio.

**Given:** Part 2 and numpy. Not allowed: MTUQ and the course package.

**Contract:**
- Inputs: `obs`, `syn`: arrays of the same shape (traces, samples), one row per trace (one station and one
  component); `dt_s`, the sample interval (s); `norm`: `"L2"`, `"L1"` or `"hybrid"` with the definitions of
  Part 2 (MTUQ's); any other `norm` raises `ValueError`. Do not change the inputs.
- Output: a dictionary with `"per_trace"` (array, one term per row), `"total"` (float, their sum), `"share"`
  (array, each term divided by the total) and `"n_eff"` (float, the participation ratio of the shares).
- Example (invented): two traces of 4 samples at `dt_s` = 0.5 whose residuals are all 1 and all 2: L2 gives
  per_trace (2, 8), total 10, share (0.2, 0.8), n_eff 1.47; hybrid gives (1, 2), 3, (0.333, 0.667), n_eff 1.8.

**Done when** `st.check("M53-T2", func=misfit_budget)` passes on 40 made-up sets of traces (residuals of known
size), all three norms, and a wrong norm name.

**Stuck?** `st.hint("M53-T2", 1)`

In [ ]:
# A colleague's cell: the L2 grid search on records where TOY_LOUD is too loud. Run it as it is.
loud_obs = toy_obs.copy()
loud_obs[TOY_CODES.index(TOY_LOUD)] *= TOY_GAIN


def colleague_l2(obs, syn, dt_s, norm):
    return {"total": float(dt_s * np.sum((obs - syn) ** 2))}


col_mech, col_mw, col_syn, col_total = toy_search(loud_obs, TOY_G, "L2", colleague_l2)
vr = 1.0 - np.sum((loud_obs.reshape(col_syn.shape) - col_syn) ** 2) / np.sum(loud_obs ** 2)
print(f"colleague: best mechanism {col_mech}, Mw {col_mw}, variance reduction {100 * vr:.1f} %")
print("colleague: 'the records are noisy, so a low variance reduction is no surprise; the search found one best "
      "mechanism, so the mechanism is well resolved'")

The **variance reduction** (glossary `variance_reduction`) printed by the colleague is $1 - \sum r^2 / \sum d^2$
over all traces together.

In [ ]:
# Before you code: the colleague's printed mechanism and variance reduction. Which station do you expect to write
# most of the colleague's misfit, and roughly what share of it? How far do you expect the colleague's mechanism to be
# from the made-up fault? Would the L1 or the hybrid norm, or leaving that station out, change the answer, and how?
st.predict("M53-T2", """

""")

In [ ]:
def misfit_budget(obs, syn, dt_s, norm="L2"):
    """Per-trace misfit terms, their total, shares and participation ratio (MTUQ's L2, L1 and hybrid norms).

    obs, syn : (traces, samples) arrays of the same shape
    dt_s     : sample interval (s)
    norm     : "L2", "L1" or "hybrid"; anything else raises ValueError
    returns  : {"per_trace": array, "total": float, "share": array, "n_eff": float}
    """
    # Step 1: the residual of every sample (record minus synthetic).
    # Step 2: one term per row with the norm's formula (mind where dt_s and the square root go).
    # Step 3: the total, the shares and n_eff = 1 / (sum of squared shares).
    # YOUR CODE HERE
    raise NotImplementedError("M53-T2: write your code here")

In [ ]:
st.check("M53-T2", func=misfit_budget)

Your budget of the colleague's fit, per station (the three components added), then the same search with the L1
and the hybrid norm, without `TOY_LOUD` and with the correct gain, each judged by the Kagan angle to the made-up
fault, by its own budget and by its variance reduction over the traces it used (about a minute).

In [ ]:
col = misfit_budget(loud_obs.reshape(col_syn.shape), col_syn, TOY_DT_S, "L2")
by_station = col["share"].reshape(len(TOY_CODES), 3).sum(axis=1)
print("colleague, share per station:", dict(zip(TOY_CODES, np.round(by_station, 3))))
print(f"colleague: n_eff {col['n_eff']:.2f} of {col['share'].size} traces")
rows = []
for label, records, norm, keep in (("colleague (L2)", loud_obs, "L2", None), ("L1", loud_obs, "L1", None),
                                   ("hybrid", loud_obs, "hybrid", None),
                                   (f"L2 without {TOY_LOUD}", loud_obs, "L2",
                                    [k for k, c in enumerate(TOY_CODES) if c != TOY_LOUD]),
                                   ("L2, correct gain", toy_obs, "L2", None)):
    mech, mw, syn, _ = toy_search(records, TOY_G, norm, misfit_budget, keep)
    keep_rows = list(range(len(TOY_CODES))) if keep is None else keep
    used = records[keep_rows].reshape(syn.shape)
    budget = misfit_budget(used, syn, TOY_DT_S, norm)
    rows.append({"run": label, "best": mech, "Mw": mw, "kagan_to_truth": round(kagan_angle(mech, TOY_MECHANISM), 1),
                 "n_eff": round(budget["n_eff"], 2), "largest share": round(float(budget["share"].max()), 3),
                 "VR (%)": round(100.0 * (1.0 - np.sum((used - syn) ** 2) / np.sum(used ** 2)), 1)})
toy_table = pd.DataFrame(rows)
print(toy_table.to_string(index=False))
print("nearest grid point to the made-up fault:",
      min(TOY_GRID, key=lambda g: kagan_angle(g, TOY_MECHANISM)), "at",
      round(min(kagan_angle(g, TOY_MECHANISM) for g in TOY_GRID), 1), "deg")



**A good-looking fit may be written by very few traces.** The left panel adds the colleague's L2 misfit
shares over each station's three components. The right compares the recovered mechanisms with the known
synthetic source. A small effective trace count means the contribution is concentrated; it is not the
number of independent stations. Different norms have different misfit definitions, so their raw totals
should not be compared as if they had the same units or scale.

**Interpret:** does the loud station dominate the colleague's budget? Which changes improve source recovery,
and does leaving one station out answer the same scientific question as correcting its gain?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].bar(TOY_CODES, by_station, color=["tab:red" if c == TOY_LOUD else "tab:blue" for c in TOY_CODES])
axes[0].set(xlabel="Synthetic station", ylabel="Share of L2 misfit", title=f"Colleague's fit: effective traces {col['n_eff']:.1f}")
axes[0].tick_params(axis="x", rotation=45)
axes[1].barh(toy_table["run"], toy_table["kagan_to_truth"], color="tab:blue")
axes[1].set(xlabel="Kagan angle to known source (°)", title="Recovery under different choices")
axes[1].invert_yaxis()
show_figure(fig, "M53_misfit_influence.png")

**Explain (3-5 sentences):** Use the printed numbers. Which station wrote the colleague's answer, with what share,
and what was the colleague's n_eff? How far was the colleague's mechanism from the made-up fault, compared with the
grid's own step (the nearest grid point)? What did L1, hybrid and leaving the station out each do, and why do the
two norms react so differently to one loud station? The colleague put the low variance reduction down to noise:
compare it with the variance reduction of the run with the correct gain (the same noise), and with those of L1 and
hybrid, whose mechanisms lie closer to the made-up fault. What can one variance reduction over all traces not tell
you that the shares can?

**Explorer (optional).** `st.widgets.mt_norm_explorer` (E14) lets you turn the gain of one made-up station up or
down, switch the norm and leave that station out, and shows each station's share of the misfit, n_eff and the
Kagan angle between the best mechanism of a coarse grid search and the made-up fault. It runs on your own
`misfit_budget`, `moment_tensor` and `kagan_angle` and computes none of them itself.

In [ ]:
try:
    explorer = st.widgets.mt_norm_explorer(misfit_budget, moment_tensor, kagan_angle, kernels=TOY_G,
                                           true_m6=TOY_TRUE, mechanism=TOY_MECHANISM, codes=TOY_CODES,
                                           dt_s=TOY_DT_S, noise_sigma=toy_sigma, seed=TOY_SEED + 1,
                                           moments_nm=[magnitude_moment(mw=m) for m in TOY_MWS])
    display(explorer)
except Exception as exc:  # noqa: BLE001 - optional cell: the module does not depend on it
    print("The explorer is not available here:", exc)

## Part 3 · Time shifts, and why they are bounded (about 60 min, then task 3)

A model Earth with flat layers (module M30) gets the travel times only roughly right: a record may arrive half a
second earlier or later than its synthetic. A misfit counted sample by sample then punishes the right mechanism,
and an inversion bends the mechanism to make up for timing. The usual cure is a **time shift** (glossary
`time_shift`; Zhu & Helmberger 1996): slide each synthetic in time and keep the shift that fits best. MTUQ does
this for each station within the bounds `time_shift_min` and `time_shift_max`. Its convention: a positive shift
means the synthetic arrived too early and is moved later.

Module M41 found shifts with the **cross-correlation** (glossary `cross_correlation`). Here you scan the misfit
itself: for every whole number of samples $k$ within the bound, delay the synthetic by $k$ samples, fill the
samples that it leaves empty with zeros, and add up the misfit with your `misfit_budget`. For the L2 norm the
smallest misfit and the largest correlation fall at the same shift as long as no part of the synthetic leaves the
window; for the L1 norm they need not.

Why a bound at all? A wave of period $T$ moved by $T/2$ has its peaks where its troughs were: it looks like itself
turned upside down. The synthetic of the opposite mechanism (every first motion flipped: the tensor $-\mathbf{m}$,
or the rake plus 180 degrees) is the right synthetic turned upside down, so a shift of part of a period can slide
it into line with the record and lower its misfit. A fit that takes such a shift jumps by part of a cycle instead
of following the arrival: **cycle skipping** (glossary `cycle_skipping`). How large the bound can become before
this matters, and how well the wrong mechanism can then fit, you measure after the task.

### M53-T3 · The best bounded time shift · core · about 50 min

**Goal:** write `best_shift(obs, syn, dt_s, max_shift_s, budget, norm="L2")`: the shift of the synthetic, within
the bound, that makes the misfit of one trace smallest.

**Given:** Part 3, numpy and your `misfit_budget` (handed in as `budget`). Not allowed: SciPy, ObsPy, MTUQ and
the course package.

**Contract:**
- Inputs: `obs` and `syn`, two arrays of the same length n (one trace each, samples `dt_s` apart); `max_shift_s`,
  the bound (s, not negative); `budget`, a function with the call and the result of your `misfit_budget`; `norm`,
  handed to `budget` unchanged. Do not change the inputs.
- Try every whole number of samples k with |k| x `dt_s` <= `max_shift_s` (allow a rounding error of 1e-9 s). The
  synthetic delayed by k samples has at sample i the value `syn[i - k]` when 0 <= i - k < n, and 0 elsewhere:
  nothing wraps around from one end to the other.
- The misfit of k is `budget(obs[None, :], delayed[None, :], dt_s, norm)["total"]`.
- Output: a tuple `(shift_s, misfit)`: k x `dt_s` of the smallest misfit (positive: the synthetic moved later)
  and that misfit. When several k give the same smallest misfit, take the one with the smallest |k|, and of two
  with the same |k| the negative one.
- Example (invented): `syn` = (0, 1, 0, 0, 0), `obs` = (0, 0, 0, 1, 0), `dt_s` = 0.5, `max_shift_s` = 1.0, L2:
  the shifts -1, -0.5, 0, 0.5, 1 s give the misfits 0.5, 1, 1, 1, 0 (at -1 s the pulse has left the trace),
  so the answer is (1.0, 0.0).

**Done when** `st.check("M53-T3", func=best_shift)` passes: made-up pulses moved by known shifts inside and
outside the bound, a pulse that a wrap-around would bring back, all three norms, and random traces compared with an
independent cross-correlation (SciPy's, which your function may not use).

**Stuck?** `st.hint("M53-T3", 1)`

In [ ]:
# Before you code: the record is the synthetic moved 0.4 s later. What shift should your function return, and with
# which sign? After the task, the opposite mechanism gets shifts within bounds from 0.5 to 4 s on band-passed made-up
# records whose periods are a few seconds. How do you expect its variance reduction to change as the bound grows, at
# what bound (as a part of the period) do you expect the largest change, and do you expect it to catch up with the
# made-up fault's?
st.predict("M53-T3", """

""")

In [ ]:
def best_shift(obs, syn, dt_s, max_shift_s, budget, norm="L2"):
    """The bounded shift of syn (whole samples, zeros fill in) with the smallest misfit of budget.

    obs, syn    : arrays of n samples, dt_s apart
    max_shift_s : bound of |shift| (s)
    budget      : function(obs_rows, syn_rows, dt_s, norm) -> dict with "total" (your misfit_budget)
    norm        : handed to budget unchanged
    returns     : (shift_s, misfit); positive shift_s = synthetic moved later
    """
    # Step 1: the largest whole number of samples K within the bound.
    # Step 2: for k = -K ... K, the synthetic delayed by k samples, with zeros where nothing comes in.
    # Step 3: the misfit of each k from budget; keep the smallest, breaking ties toward small |k|, then negative k.
    # YOUR CODE HERE
    raise NotImplementedError("M53-T3: write your code here")

In [ ]:
st.check("M53-T3", func=best_shift)

Your function on the made-up network. Every made-up station gets a travel-time error, a delay drawn between
`-TOY_DELAY_S` and `+TOY_DELAY_S` (the same for its three components), as a model Earth that is a little too fast
or too slow toward it would make. First the least squares of task 1 without shifts; then one round of your shifts:
for every trace the best shift of the least-squares synthetic within `TOY_MAX_SHIFT_S`, each trace's elementary
seismograms moved by its shift, and the least squares again.

In [ ]:
def move(trace, k):
    """A trace (last axis = samples) moved k samples later, zeros coming in at the open end."""
    trace = np.asarray(trace, dtype=float)
    if k == 0:
        return trace.copy()
    pad = np.zeros(trace.shape[:-1] + (abs(k),))
    if k > 0:
        return np.concatenate([pad, trace[..., :-k]], axis=-1)
    return np.concatenate([trace[..., -k:], pad], axis=-1)


delay_gen = np.random.default_rng(TOY_SEED + 3)
toy_delays = np.round(delay_gen.uniform(-TOY_DELAY_S, TOY_DELAY_S, len(TOY_CODES)) / TOY_DT_S).astype(int)
late_obs = np.stack([move(toy_obs[s], int(toy_delays[s])) for s in range(len(TOY_CODES))])
print("made-up delays (s):", dict(zip(TOY_CODES, np.round(toy_delays * TOY_DT_S, 1))))

m_plain, _ = invert_mt(toy_Gmat, late_obs.reshape(-1), toy_sigma)
syn_plain = (toy_Gmat @ m_plain).reshape(late_obs.shape)
found = np.zeros((len(TOY_CODES), 3))
for s in range(len(TOY_CODES)):
    for c in range(3):
        found[s, c] = best_shift(late_obs[s, c], syn_plain[s, c], TOY_DT_S, TOY_MAX_SHIFT_S, misfit_budget)[0]
steps = np.round(found / TOY_DT_S).astype(int)
moved_G = np.stack([np.stack([move(TOY_G[s, c].T, int(steps[s, c])).T for c in range(3)])
                    for s in range(len(TOY_CODES))])
m_shifted, _ = invert_mt(moved_G.reshape(-1, 6), late_obs.reshape(-1), toy_sigma)
for label, m_hat in (("no shifts", m_plain), ("one round of shifts", m_shifted)):
    print(f"{label:20s} largest |error| / M0 = {np.abs(m_hat - TOY_TRUE).max() / TOY_M0:.3f}")
print("your shifts (s), rows = stations, columns Z R T:\n",
      pd.DataFrame(found, index=TOY_CODES, columns=list("ZRT")).round(2).to_string())

Now the bound. Real records are fitted in a narrow band (Part 6: `BAND_HZ`), where a wave is a few cycles of one
period rather than one sharp pulse. The next cell band-passes the delayed records and the synthetics of the
made-up fault alike with `TOY_BAND_HZ` (a numpy band-pass in the frequency domain, applied to both, so its phase
cannot matter); then, for the made-up fault and for its opposite ($-\mathbf{m}$: every first motion flipped), it
adds up the L2 misfit of all 24 traces after your best shifts, for the bounds `TOY_BOUNDS_S`, and prints it as the
variance reduction of Part 2 (100 % = a perfect fit, 0 % = no better than no synthetic), with the period at which
the band-passed records carry most of their energy (the peak of their average amplitude spectrum).

Last, it measures how much the band-passed synthetics look like their own negative after a shift: their
correlation with themselves at every lag (added over the 24 traces and divided by their energy; 1 = a perfect
copy, -1 = a perfect upside-down copy), and the lag where it is most negative. For a record and a synthetic of the
same energy whose correlation after a shift is $c$, the L2 misfit after that shift is $2E(1-c)$, with $E$ the
energy, and the variance reduction is $2c - 1$.

In [ ]:
def band_limit(traces, band_hz, dt_s):
    """The traces (last axis = samples) with only the frequencies inside band_hz kept (cosine-tapered edges)."""
    spectrum = np.fft.rfft(traces, axis=-1)
    f = np.fft.rfftfreq(traces.shape[-1], dt_s)
    lo, hi = band_hz
    gain = np.clip((f - 0.5 * lo) / (0.5 * lo), 0.0, 1.0) * np.clip((1.5 * hi - f) / (0.5 * hi), 0.0, 1.0)
    return np.fft.irfft(spectrum * np.sin(0.5 * np.pi * gain) ** 2, n=traces.shape[-1], axis=-1)


syn_true = band_limit(np.einsum("scnj,j->scn", TOY_G, TOY_TRUE), TOY_BAND_HZ, TOY_DT_S)
narrow_obs = band_limit(late_obs, TOY_BAND_HZ, TOY_DT_S)
energy = TOY_DT_S * np.sum(narrow_obs ** 2)
rows = []
for bound in TOY_BOUNDS_S:
    totals = {}
    for label, sign in (("made-up fault", 1.0), ("opposite", -1.0)):
        totals[label] = sum(best_shift(narrow_obs[s, c], sign * syn_true[s, c], TOY_DT_S, bound, misfit_budget)[1]
                            for s in range(len(TOY_CODES)) for c in range(3))
    rows.append({"bound_s": bound, "VR made-up fault (%)": 100.0 * (1.0 - totals["made-up fault"] / energy),
                 "VR opposite (%)": 100.0 * (1.0 - totals["opposite"] / energy)})
bound_table = pd.DataFrame(rows)
print(bound_table.round(1).to_string(index=False))
spectrum = np.abs(np.fft.rfft(narrow_obs, axis=-1)).mean(axis=(0, 1))
freqs = np.fft.rfftfreq(narrow_obs.shape[-1], TOY_DT_S)
print(f"the band-passed records' energy peaks at {freqs[np.argmax(spectrum)]:.3f} Hz, a period of "
      f"{1.0 / freqs[np.argmax(spectrum)]:.2f} s")
flat = syn_true.reshape(-1, syn_true.shape[-1])
auto = sum(np.correlate(trace, trace, mode="full") for trace in flat)
auto = auto / auto[flat.shape[-1] - 1]
lags_s = (np.arange(auto.size) - (flat.shape[-1] - 1)) * TOY_DT_S
most = int(np.argmin(np.where(lags_s > 0, auto, np.inf)))
print(f"the band-passed synthetics look most like their own negative when moved by {lags_s[most]:.1f} s "
      f"(either way): correlation {auto[most]:.2f}")



**Time shifts can absorb model error and also hide a wrong source.** The left panel compares your recovered
shifts with the imposed synthetic delays. Each symbol is one component at a station. The right shows how
loosening the allowed shift changes the fit of the correct and opposite source. Both curves use the same
band-limited observations, so their variance reductions are comparable here.

**Interpret:** where do shifts approach the edge of their bound? At what bound can the wrong polarity start
to fit a different oscillation? Relate the change to the period and correlation curve printed above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
for component, marker in enumerate(("o", "s", "^")):
    axes[0].scatter(toy_delays * TOY_DT_S, found[:, component], marker=marker, label="ZRT"[component], s=30)
visual_limit = max(TOY_MAX_SHIFT_S, TOY_DELAY_S) * 1.15
axes[0].plot([-visual_limit, visual_limit], [-visual_limit, visual_limit], "k--", lw=0.8)
axes[0].set(xlabel="Imposed delay (s)", ylabel="Your shift (s)", xlim=(-visual_limit, visual_limit),
            ylim=(-visual_limit, visual_limit), title="Known versus recovered timing")
axes[0].legend(fontsize=8)
for column, label in (("VR made-up fault (%)", "Known source"), ("VR opposite (%)", "Opposite source")):
    axes[1].plot(bound_table["bound_s"], bound_table[column], "o-", label=label)
axes[1].set(xlabel="Allowed |shift| (s)", ylabel="Variance reduction (%)", title="The cost of a permissive bound")
axes[1].legend(fontsize=8)
show_figure(fig, "M53_time_shift_tradeoff.png")

**Explain (3-5 sentences):** Use the printed numbers. How much did one round of shifts improve the tensor, how
close are your shifts to the made-up delays (mind the sign), and where do they differ most? Between which two
bounds did the variance reduction of the opposite mechanism change most, what part of the printed period is that,
and how high did it get against the made-up fault's? Do the printed lag and correlation at which the synthetics
look most like their own negative agree with that, in where the change happens (mind the made-up delays) and in
how high it goes ($2c - 1$)? What does this say about a safe bound, and what part of the shortest period of
`BAND_HZ` is the bound of Part 6 (`MAX_SHIFT_S`)? Was your prediction right?

## Part 4 · What kind of source is a tensor? The lune and the double-couple planes (about 60 min, then task 4)

A tensor from a free inversion is rarely a pure double couple. Its three eigenvalues
$\lambda_1 \ge \lambda_2 \ge \lambda_3$ say what it is, and Tape & Tape (2012) put every possible set of them on
one map, the **lune** (glossary `lune`): with $|\lambda| = \sqrt{\lambda_1^2 + \lambda_2^2 + \lambda_3^2}$,

$$\delta = 90^\circ - \arccos\frac{\lambda_1 + \lambda_2 + \lambda_3}{\sqrt{3}\,|\lambda|},\qquad
\gamma = \arctan\frac{-\lambda_1 + 2\lambda_2 - \lambda_3}{\sqrt{3}\,(\lambda_1 - \lambda_3)} .$$

The lune latitude $\delta$ runs from $-90^\circ$ (an implosion) to $+90^\circ$ (an explosion); the lune longitude
$\gamma$ from $-30^\circ$ to $+30^\circ$ (the two CLVDs); a double couple sits at $\gamma = \delta = 0$. Tape &
Tape's scalar moment is $M_0 = |\lambda| / \sqrt{2}$, which is the usual $M_0$ for a double couple. The tensor's
eigenvectors give its **P and T axes** (glossary `p_t_axes`): T belongs to $\lambda_1$, P to $\lambda_3$, as in
M50. The double couple with the same axes has the two **nodal planes** (glossary `nodal_plane`) whose normal and
slip vectors are

$$\mathbf{n} = (\mathbf{t} + \mathbf{p})/\sqrt{2},\quad \mathbf{s} = (\mathbf{t} - \mathbf{p})/\sqrt{2}
\qquad\text{and}\qquad \mathbf{n} = (\mathbf{t} - \mathbf{p})/\sqrt{2},\quad \mathbf{s} = (\mathbf{t} +
\mathbf{p})/\sqrt{2},$$

up to a common sign. Your `fault_vectors` of M50 went from (strike, dip, rake) to $\mathbf{n}$ and $\mathbf{s}$
in north-east-down; here you go back. Mind the frame: `m6` is in up-south-east (Part 0), so its eigenvectors are
too.

### M53-T4 · Lune coordinates and double-couple planes of a tensor · core · about 75 min

**Goal:** write `tensor_summary(m6)`: the eigenvalues, the scalar moment, the lune coordinates and the two
double-couple planes of a moment tensor.

**Given:** Part 4, your `fault_vectors` of M50 as a model of the conventions, and numpy (`numpy.linalg.eigh` is
fine). Not allowed: MTUQ, pyrocko, ObsPy (its beach-ball and MoPaD code turn tensors into planes), MoPaD and the
course package (the check compares with MTUQ's and pyrocko's).

**Contract:**
- Input: `m6`, six numbers (Mrr, Mtt, Mpp, Mrt, Mrp, Mtp) in N m, up-south-east as in Part 0. Do not change it.
- Output: a dictionary with `"eigenvalues"` (array of 3, N m, largest first), `"m0_nm"` (float, Tape & Tape's
  $|\lambda|/\sqrt{2}$), `"gamma_deg"` and `"delta_deg"` (floats, the lune coordinates of Part 4) and `"planes"`
  (a list of two tuples (strike, dip, rake) in degrees, strike in [0, 360), dip in [0, 90], rake in (-180, 180],
  in either order), the planes of the double couple that has the tensor's T and P axes.
- Example (invented): the tensor of strike 40, dip 70, rake 20 (your `moment_tensor` in USE) with M0 = 1e13 N m
  gives eigenvalues (1e13, 0, -1e13), `m0_nm` 1e13, gamma 0, delta 0, and the planes (40, 70, 20) and
  (302.9, 71.3, 158.8).

**Done when** `st.check("M53-T4", func=tensor_summary)` passes on 60 made-up tensors of every source type
(double couples, CLVDs, tensors with a change of volume): eigenvalues from lune coordinates chosen by the check,
eigenvectors of double couples of pyrocko, whose planes are the reference.

**Stuck?** `st.hint("M53-T4", 1)`

In [ ]:
# Before you code: where on the lune do you expect your least-squares tensors of Part 1 (free, and deviatoric with two
# stations silent)? Will their planes lie close to the made-up fault, and will a free inversion with 5 % noise put
# the tensor exactly on the double couple?
st.predict("M53-T4", """

""")

In [ ]:
def tensor_summary(m6):
    """Eigenvalues, M0, lune coordinates and double-couple planes of a tensor (USE order, N m).

    m6      : (Mrr, Mtt, Mpp, Mrt, Mrp, Mtp) in N m
    returns : {"eigenvalues": (3,) largest first, "m0_nm": float, "gamma_deg": float, "delta_deg": float,
               "planes": [(strike, dip, rake), (strike, dip, rake)]}
    """
    # Step 1: the symmetric 3 x 3 tensor, its eigenvalues and eigenvectors, sorted largest first.
    # Step 2: |lambda|, M0 and the two lune angles of Part 4.
    # Step 3: t and p (eigenvectors of the largest and smallest) in north-east-down; for each of the two (n, s)
    #         pairs, make n point upward (flip both if needed), then strike, dip and rake as fault_vectors of M50
    #         would have them.
    # YOUR CODE HERE
    raise NotImplementedError("M53-T4: write your code here")

In [ ]:
st.check("M53-T4", func=tensor_summary)

Your function on the tensors of Parts 1-3: the made-up fault itself, the free least squares with all stations,
the deviatoric one with two stations silent, the colleague's mechanism of Part 2 and the least squares after one
round of shifts. The Kagan angle (your function of M51) is measured from the first plane to the made-up fault.

In [ ]:
m_all, _ = invert_mt(toy_Gmat, toy_d, toy_sigma)
m_dev, _ = invert_mt(toy_Gmat, gappy.reshape(-1), toy_sigma, deviatoric=True)
summary_rows = []
for label, m6 in (("made-up fault", TOY_TRUE), ("free, all stations", m_all), ("deviatoric, two silent", m_dev),
                  ("colleague (Part 2)", six(moment_tensor(*col_mech, magnitude_moment(mw=col_mw), frame="USE"))),
                  ("after shifts (Part 3)", m_shifted)):
    info = tensor_summary(m6)
    summary_rows.append({"tensor": label, "Mw": round(magnitude_moment(m0_nm=info["m0_nm"]), 3),
                         "gamma_deg": round(info["gamma_deg"], 2), "delta_deg": round(info["delta_deg"], 2),
                         "plane 1": tuple(np.round(info["planes"][0], 1)),
                         "plane 2": tuple(np.round(info["planes"][1], 1)),
                         "kagan_to_truth": round(kagan_angle(info["planes"][0], TOY_MECHANISM), 1)})
print(pd.DataFrame(summary_rows).to_string(index=False))

**Source type is another dimension of the interpretation.** These are the lune coordinates returned by your
function, plotted on rectangular axes to make their numerical values readable. The star at (0,0) is a pure
double couple. This coordinate plot is not an equal-area projection, and distances on the page are not
Kagan angles. The right panel compares the double-couple orientations with the known synthetic fault.

**Interpret:** can a tensor lie close to the double-couple point and still have the wrong orientation? Why
does a zero-trace constraint control one source-type coordinate without fixing the mechanism?

In [ ]:
visual_summary = pd.DataFrame(summary_rows)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
axes[0].plot(0, 0, "k*", ms=14, label="Double couple")
for i, row in visual_summary.iterrows():
    axes[0].scatter(row["gamma_deg"], row["delta_deg"], s=35)
    axes[0].annotate(str(i + 1), (row["gamma_deg"], row["delta_deg"]), xytext=(4, 4), textcoords="offset points")
axes[0].axhline(0, color="0.7", lw=0.7)
axes[0].axvline(0, color="0.7", lw=0.7)
axes[0].set(xlabel="Lune longitude γ (°)", ylabel="Lune latitude δ (°)", title="Source-type coordinates (row numbers)")
axes[1].barh([f"{i + 1}. {v}" for i, v in enumerate(visual_summary["tensor"])], visual_summary["kagan_to_truth"], color="tab:blue")
axes[1].set(xlabel="Kagan angle to known source (°)", title="Double-couple orientation")
axes[1].invert_yaxis()
show_figure(fig, "M53_source_type_and_orientation.png")

**Explain (3-5 sentences):** Use the printed rows. How far from the double couple did noise and gaps put the free
and the deviatoric tensors (gamma and delta), and how large were their Kagan angles? Why is delta of the
deviatoric tensor what it is? What does the Kagan angle of the colleague's row add to what Part 2 showed? Was your
prediction right?

## Part 5 · The Sangju mainshock: records, Green's functions and a first inversion (about 90 min, then task 5)

From here on the records are real: the ML 3.9 earthquake near Sangju on 21 July 2019 (KMA catalog), which you
picked in M20, whose first motions you inverted in M51 and whose synthetics you computed in M52. This part prepares
everything that MTUQ (glossary `mtuq`; Thurin et al. 2025; Silwal & Tape 2016; Alvizuri et al. 2018) reads, and
makes a first, linear inversion with your own `invert_mt`:

1. the stations where you picked both P and S, with three components and a response file;
2. the records as ground displacement in up, radial, transverse (`Z`, `R`, `T`; glossary `radial_transverse`),
   written as SAC files that MTUQ can read;
3. a **Green's function database** (glossary `greens_function_database`) from PyFK in the folder layout of the FK
   programs (Zhu & Rivera 2002), which `mtuq.open_db(..., format="FK")` reads;
4. an **origin time** (glossary `origin_time`) from your P picks, and a weights file in which your picks place
   MTUQ's windows;
5. an **amplitude check** (glossary `amplitude_consistency`, your task 5) before any fit;
6. a least-squares tensor at every trial depth.

The instrument responses are StationMetadata by Byeong-Seok Ahn (PKNU; Ahn et al. 2024).

In [ ]:
catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
EVENT_FOLDER = ORIGIN_KMA.strftime("%Y%m%d%H%M%S")
SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER / BAND
XML_DIR = st.data.root() / "metadata" / "stationxml"
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no {BAND} waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data "
                            "with STP first (module M03), or set DATASET in the parameters cell to your folder.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. MTUQ's windows come from your own picks: finish task M20-T5 "
                            "of module M20 first (its picker saves the file).")
print(f"KMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), {EV_LAT:.2f} N {EV_LON:.2f} E, depth "
      f"{float(event['depth_km']):g} km, ML {float(event['ml']):.1f}; source: {catalog.attrs.get('source', 'KMA')}")

picks = pd.read_csv(PICKS_CSV, dtype=str)
picks["time"] = [UTCDateTime(t) for t in picks["time_utc"]]
picks = picks[[0.0 <= (t - ORIGIN_KMA) <= 120.0 for t in picks["time"]]]
picks = picks.sort_values("time").drop_duplicates(["station", "phase"])
station_rows = []
for name, mine in picks.groupby("station"):
    times = dict(zip(mine["phase"], mine["time"]))
    files = sorted(SAC_DIR.glob(f"*.{name}.{BAND}?.sac"))
    if not {"P", "S"} <= set(times) or len(files) != 3 or not (XML_DIR / f"{name}.xml").is_file():
        print(f"{name}: left out (needs a P and an S pick, three {BAND} components and a response file)")
        continue
    head = read(str(files[0]), headonly=True)[0].stats.sac
    dist_m, az, baz = gps2dist_azimuth(EV_LAT, EV_LON, float(head.stla), float(head.stlo))
    station_rows.append({"station": name, "latitude": float(head.stla), "longitude": float(head.stlo),
                         "distance_km": dist_m / 1000.0, "azimuth_deg": az, "back_azimuth_deg": baz,
                         "p_time": times["P"], "s_time": times["S"]})
stations = pd.DataFrame(station_rows).sort_values("distance_km").reset_index(drop=True)
stations["offset_km"] = stations["distance_km"].round().astype(int)
print(stations[["station", "distance_km", "offset_km", "azimuth_deg", "back_azimuth_deg"]].round(2)
      .to_string(index=False))

**The records.** As in M10 and M52: demean, 5 % Hann taper and response removal with `PRE_FILT_HZ` and
`water_level=None`, but to ground **displacement** in metres this time (some channels print the evalresp warning
about sensitivities that M03 explained). Then the two horizontals are turned into radial and transverse with the
back azimuth (ObsPy's `rotate`), the records are decimated by `DECIMATE` (ObsPy's `decimate`, which low-passes
first) and written to `results/M53_records/` with the station and event coordinates in their SAC headers. Nothing
is band-passed yet: MTUQ does that later, in the same way for records and Green's functions.

In [ ]:
REC_DIR = Path(RESULT_DIR) / "M53_records"
if REC_DIR.exists():
    shutil.rmtree(REC_DIR)
REC_DIR.mkdir(parents=True)
for row in stations.itertuples():
    three = read(str(SAC_DIR / f"*.{row.station}.{BAND}?.sac"))
    three.detrend("demean").taper(max_percentage=0.05, type="hann")
    three.remove_response(inventory=read_inventory(str(XML_DIR / f"{row.station}.xml")), output="DISP",
                          pre_filt=PRE_FILT_HZ, water_level=None)
    three.rotate("NE->RT", back_azimuth=row.back_azimuth_deg)
    three.decimate(DECIMATE)
    for trace in three:
        trace.stats.sac.update({"stla": row.latitude, "stlo": row.longitude, "evla": EV_LAT, "evlo": EV_LON,
                                "evdp": float(event["depth_km"])})
        trace.write(str(REC_DIR / f"{trace.id}.sac"), format="SAC")
print(f"{len(list(REC_DIR.glob('*.sac')))} SAC files in {short_path(REC_DIR)}, "
      f"{1.0 / three[0].stats.delta:g} samples per second")

**The Green's function database.** The FK programs store, for one model, one source depth and one distance,
the ground motion of ten **fundamental sources** (three double couples called SS, DS, DD and an explosion EP, each
seen on Z and R, and SS and DS on T); MTUQ combines them for any tensor and any azimuth with fixed formulas
(`mtuq/greens_tensor/FK.py`, read them). PyFK computes the same ten internally, but `fk.run` hands back a
synthetic for one source. Four runs per depth recover all ten exactly, because PyFK's radiation coefficients for a
moment tensor (`pyfk/config/radiats.py`, `mt_radiat`, read it) single each one out:

| run | source (north-east-down elements, 1e20 dyne-cm) | azimuth | gives |
|---|---|---|---|
| `ne_nd` | $M_{ne} = M_{nd} = 1$ | 0 | Z = -ZDS, R = -RDS, T = TSS |
| `ne_45` | $M_{ne} = 1$ | 45 | Z = -ZSS, R = -RSS |
| `ed_dd` | $M_{ed} = M_{dd} = 1$ | 0 | Z = ZDD/3, R = RDD/3, T = TDS |
| `explosion` | PyFK's `ep` source | 0 | Z = ZEP, R = REP |

This module's searches use double couples and trace-free tensors, which never read ZEP and REP. Module M52 found
that PyFK's samples are ground velocity in cm/s for a moment of 1e20 dyne-cm; the database holds their time
integral (a running sum times `FK_DT_S`), displacement, because the records are displacement. MTUQ multiplies the
files by 1e-15 to turn cm per 1e20 dyne-cm into metres per N m. Every distance is rounded to whole kilometres (the
FK file names), and all distances of one depth go into one PyFK call (Part 6 of M52: PyFK's wavenumber step
depends on the largest distance of a call). `DEPTHS_KM` holds whole kilometres and leaves out 15, the depth of a
layer boundary in kim1983.

In [ ]:
LAYERS = fk.model_layers(st.data.velocity_model(VELOCITY_MODEL))
PULSE = fk.trapezoid_stf(STF_DURATION_S, 0.5, FK_DT_S)
OFFSETS = sorted(set(int(v) for v in stations["offset_km"]))
DB_DIR = Path(RESULT_DIR) / "M53_greens" / VELOCITY_MODEL
FK_SOURCES = {"ne_nd": ([1e20, 0, 1, 1, 0, 0, 0], "dc", 0.0), "ne_45": ([1e20, 0, 1, 0, 0, 0, 0], "dc", 45.0),
              "ed_dd": ([1e20, 0, 0, 0, 0, 1, 1], "dc", 0.0), "explosion": ([1e20], "ep", 0.0)}
FK_FILES = {"0": ("ed_dd", "z", 3.0), "1": ("ed_dd", "r", 3.0), "2": ("ed_dd", "t", 0.0),     # ZDD RDD (TDD = 0)
            "3": ("ne_nd", "z", -1.0), "4": ("ne_nd", "r", -1.0), "5": ("ed_dd", "t", 1.0),   # ZDS RDS TDS
            "6": ("ne_45", "z", -1.0), "7": ("ne_45", "r", -1.0), "8": ("ne_nd", "t", 1.0),   # ZSS RSS TSS
            "9": ("explosion", "t", 0.0), "a": ("explosion", "z", 1.0), "b": ("explosion", "r", 1.0)}  # ZEP REP


def build_depth(depth_km):
    """The FK files of one depth: <DB_DIR>/<model>_<depth>/<offset>.grn.<0-9, a, b>, displacement."""
    runs = {name: fk.run(LAYERS, source_depth_km=depth_km, distances_km=OFFSETS, azimuths_deg=az, source=src,
                         src_type=kind, dt_s=FK_DT_S, npt=FK_NPT, stf=PULSE, quiet=True,
                         out_dir=Path(RESULT_DIR) / "M53_pyfk" / f"{depth_km}_{name}")
            for name, (src, kind, az) in FK_SOURCES.items()}
    folder = DB_DIR / f"{VELOCITY_MODEL}_{depth_km}"
    folder.mkdir(parents=True, exist_ok=True)
    first = runs["ne_nd"]
    for i, offset in enumerate(OFFSETS):
        for ext, (name, comp, factor) in FK_FILES.items():
            velocity = factor * runs[name].component(comp)[i]
            trace = Trace(data=(np.cumsum(velocity) * FK_DT_S).astype(np.float32))
            trace.stats.delta = FK_DT_S
            trace.stats.starttime = UTCDateTime(0) + float(first.b_s[i])
            trace.stats.sac = {"t1": float(first.t_p_s[i]), "t2": float(first.t_s_s[i]), "dist": float(offset)}
            trace.write(str(folder / f"{offset}.grn.{ext}"), format="SAC")
    return first


if DB_DIR.exists():
    shutil.rmtree(DB_DIR)
first_arrivals = {}
for depth in DEPTHS_KM:
    run = build_depth(depth)
    first_arrivals[depth] = dict(zip(OFFSETS, run.t_p_s))
digest = hashlib.sha256()
for path in sorted(DB_DIR.rglob("*.grn.*")):
    digest.update(path.name.encode() + path.read_bytes())
print(f"database {short_path(DB_DIR)}: {len(DEPTHS_KM)} depths x {len(OFFSETS)} distances x 12 files; "
      f"SHA-256 of all files {digest.hexdigest()[:16]}...")

Before trusting the database, test it. MTUQ reads it (`open_db`), places each station, and makes the synthetic
of KIGAM's plane A with KIGAM's Mw at `REFERENCE_DEPTH_KM`; PyFK makes the same synthetic directly (a four-number
source, one call with the same distances and each station's azimuth), integrated in the same way. Both are
band-passed with `BAND_HZ` (causal, 4 poles) and compared with the **correlation coefficient** (M41) and the
ratio of their largest values.

In [ ]:
import mtuq
from mtuq import open_db
from mtuq.event import MomentTensor, Origin
from mtuq.grid import DoubleCoupleGridRegular
from mtuq.grid_search import grid_search
from mtuq.misfit import Misfit
from mtuq.process_data import ProcessData
from mtuq.util.math import to_mij, to_Mw, to_rho
from obspy.signal.filter import bandpass

db = open_db(str(DB_DIR), format="FK", model=VELOCITY_MODEL)
all_ids = [f"{name}." for name in stations["station"]]
records = mtuq.read(str(REC_DIR / "*.sac"), format="sac", event_id=EVENT_FOLDER, station_id_list=all_ids,
                    tags=["units:m", "type:displacement"])
records.sort_by_distance()
ref_origin = Origin({"time": ORIGIN_KMA, "latitude": EV_LAT, "longitude": EV_LON,
                     "depth_in_m": 1000.0 * REFERENCE_DEPTH_KM, "id": f"{EVENT_FOLDER}_ref"})
kigam_mt = MomentTensor(to_mij(to_rho(KIGAM_MW), 0.0, 0.0, KIGAM_PLANE_A[0], KIGAM_PLANE_A[2],
                               np.cos(np.radians(KIGAM_PLANE_A[1]))))
via_mtuq = db.get_greens_tensors(records.get_stations(), [ref_origin]).get_synthetics(kigam_mt,
                                                                                     components=["Z", "R", "T"])
order = [stations.index[stations["station"] == f"{s.network}.{s.station}"][0] for s in records.get_stations()]
direct = fk.run(LAYERS, source_depth_km=REFERENCE_DEPTH_KM, distances_km=stations["offset_km"].iloc[order].tolist(),
                azimuths_deg=stations["azimuth_deg"].iloc[order].tolist(), source=[KIGAM_MW, *KIGAM_PLANE_A],
                dt_s=FK_DT_S, npt=FK_NPT, stf=PULSE, out_dir=Path(RESULT_DIR) / "M53_pyfk" / "direct", quiet=True)
test_rows = []
for i, stream in enumerate(via_mtuq):
    for comp in "ZRT":
        mine = stream.select(component=comp)[0]
        times = float(mine.stats.starttime - ORIGIN_KMA) + mine.stats.delta * np.arange(mine.stats.npts)
        other = np.interp(times, direct.times(i), np.cumsum(direct.component(comp.lower())[i]) * FK_DT_S * 1e-2,
                          left=0.0, right=np.nan)
        keep = np.isfinite(other)
        a = bandpass(mine.data[keep], *BAND_HZ, 1.0 / mine.stats.delta, corners=4, zerophase=False)
        b = bandpass(other[keep], *BAND_HZ, 1.0 / mine.stats.delta, corners=4, zerophase=False)
        test_rows.append({"station": records.get_stations()[i].station, "comp": comp,
                          "cc": round(float(np.corrcoef(a, b)[0, 1]), 4),
                          "ratio": round(float(np.abs(a).max() / np.abs(b).max()), 4)})
test_table = pd.DataFrame(test_rows)
print(test_table.pivot(index="station", columns="comp", values=["cc", "ratio"]).to_string())

**An origin time from your picks.** KMA's origin time is given to the whole second. With PyFK's first-P time
at `REFERENCE_DEPTH_KM` (the KMA depth), every P pick gives one estimate, pick minus travel time; their median is
the origin time used from here on (`ORIGIN_USED`). MTUQ places each window at that origin time plus the travel
time in the **weights file**, `results/M53_weights.dat` (MTUQ's `pick_type="user_supplied"`, the CAPUAF format:
one line per station with its distance, five weights, and the P and S travel times in columns 8 and 10). With
`window_type="surface_wave"` MTUQ opens the window `0.3 x WINDOW_S` before the S time.

In [ ]:
estimates = np.array([(row.p_time - ORIGIN_KMA) - first_arrivals[REFERENCE_DEPTH_KM][int(row.offset_km)]
                      for row in stations.itertuples()])
ORIGIN_USED = ORIGIN_KMA + float(np.median(estimates))
print("P pick minus PyFK travel time (s):", dict(zip(stations["station"], np.round(estimates, 2))))
print(f"origin time used: {ORIGIN_USED} (KMA {ORIGIN_KMA} plus the median, {np.median(estimates):+.2f} s)")


def write_weights(path, table):
    """MTUQ's weights file for the stations of table: Z, R, T of the surface-wave window weighted 1."""
    lines = []
    for row in table.itertuples():
        net, code = row.station.split(".")
        lines.append(f"{EVENT_FOLDER}.{net}.{code}.. {row.distance_km:.2f} 0 0 1 1 1 "
                     f"{row.p_time - ORIGIN_USED:.3f} 0 {row.s_time - ORIGIN_USED:.3f} 0 0 0")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")

**The amplitude check.** An L2 or hybrid misfit compares amplitudes, so a record whose size is wrong (a wrong
gain in the response file, a unit error) takes over the misfit, as in Part 2. Before any fit, compare the peak
displacements of the stations. Ground motion falls with distance (glossary `geometric_spreading`): roughly as
$1/r$ for body waves. So $\log_{10}(A\,r)$, the peak $A$ times the distance $r$, should scatter about one common
level; the radiation pattern and the errors of a flat-layered model make it scatter by a few tenths. The median of
all stations is that level, because the median, unlike the mean, does not move when one station is far off. A
station whose deviation from it is larger than `QC_MAX_DEV` (in powers of ten: 0.6 is a factor of 4) is left out.
The peak is taken in the band and the window that MTUQ will fit: the largest |displacement| of the three
components, band-passed with `BAND_HZ` (causal, 4 poles, as MTUQ does), between `0.3 x WINDOW_S` before and
`0.7 x WINDOW_S` after your S pick.

### M53-T5 · An amplitude check, and the least-squares tensor of the Sangju mainshock · core · about 100 min

**Goal:** write `amplitude_check(peak_m, distance_km, max_dev)`, apply it to the Sangju records, and invert the
kept records with your `invert_mt` at every trial depth; report the result in a dictionary `sangju_r2`.

**Given:** the amplitude paragraph above and numpy; for the report, the cells below (they call your functions and
write `results/M53_amplitudes.csv`, `results/M53_lsq.csv` and `results/M53_lsq_traces.csv`). Not allowed in
`amplitude_check`: SciPy, pandas, MTUQ, ObsPy and the course package.

**Contract (the function):**
- Inputs: `peak_m`, the peak amplitudes (m) of n stations; `distance_km`, their distances (km); `max_dev`, the
  largest accepted |deviation| (powers of ten). Do not change the inputs.
- A station with a peak or a distance that is not a positive finite number has the deviation NaN and is not kept;
  the level is the median of $\log_{10}(A\,r)$ over the other stations, and each of those has the deviation
  $\log_{10}(A\,r)$ minus the level.
- Output: a dictionary with `"deviation"` (array of n), `"keep"` (boolean array of n: |deviation| <= `max_dev`)
  and `"level"` (float).
- Example (invented): peaks (1e-6, 2e-6, 1e-6, 8e-5) m at (10, 10, 20, 10) km with `max_dev` = 0.6 give the level
  log10(2e-5) = -4.699, the deviations (-0.301, 0, 0, 1.602) and keep (True, True, True, False).

**Contract (the report):** `sangju_r2 = {"rejected": [...], "max_dev": ..., "amplitudes_sha256": ...,
"lsq_sha256": ..., "lsq_depth_km": ..., "lsq_planes": [[...], [...]], "kagan_lsq_kigam": ..., "n_eff_lsq": ...}`:
the stations your check left out (NET.STA), the `max_dev` you used, the SHA-256 fingerprints of
`results/M53_amplitudes.csv` and `results/M53_lsq.csv`, the trial depth with the smallest least-squares misfit,
your two planes of that depth's tensor, the Kagan angle from its first plane to KIGAM's plane A, and n_eff of that
fit over its traces.

**Done when** `st.check("M53-T5", func=amplitude_check, report=sangju_r2, ...)` (the cell below) passes: made-up
networks with known deviations (one or two stations off by known factors, gaps and zeros), then your files: the
check recomputes your decisions with your function, your planes with your `tensor_summary`, the Kagan angle with
your `kagan_angle` and n_eff from your trace file, and refuses stale fingerprints. It prints no value of the Sangju
data.

**Stuck?** `st.hint("M53-T5", 1)`

In [ ]:
# Before you code: think of Part 2 and of what you know about these stations from M52. Do you expect a station of
# the Sangju network to fail the amplitude check, and would leaving it out matter? How far from KIGAM's mechanism
# (in Kagan angle) do you expect a free least-squares tensor without time shifts, and at which depth?
st.predict("M53-T5", """

""")

In [ ]:
def amplitude_check(peak_m, distance_km, max_dev):
    """Deviation of log10(peak x distance) from the median of all stations, and which stations to keep.

    peak_m      : (n,) peak amplitudes (m)
    distance_km : (n,) distances (km)
    max_dev     : largest accepted |deviation| (powers of ten)
    returns     : {"deviation": (n,) array, NaN for unusable stations; "keep": (n,) bool; "level": float}
    """
    # Step 1: which stations have a positive finite peak and distance.
    # Step 2: log10(peak x distance) of those, and its median (the level).
    # Step 3: deviations from the level; keep where |deviation| <= max_dev.
    # YOUR CODE HERE
    raise NotImplementedError("M53-T5: write your code here")

Your check on the Sangju records. The table is saved to `results/M53_amplitudes.csv`; the stations you keep go
into the weights file.

In [ ]:
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


peaks = []
for row in stations.itertuples():
    three = read(str(REC_DIR / f"*.{row.station.split('.')[1]}.*.sac"))
    three.filter("bandpass", freqmin=BAND_HZ[0], freqmax=BAND_HZ[1], corners=4, zerophase=False)
    start = row.s_time - 0.3 * WINDOW_S
    peaks.append(max(float(np.abs(tr.slice(start, start + WINDOW_S).data).max()) for tr in three))
amps = stations[["station", "distance_km"]].copy()
amps["peak_m"] = peaks
qc = amplitude_check(amps["peak_m"].to_numpy(), amps["distance_km"].to_numpy(), QC_MAX_DEV)
amps["deviation"], amps["keep"] = qc["deviation"], qc["keep"]
AMP_CSV = Path(RESULT_DIR) / "M53_amplitudes.csv"
amps.to_csv(AMP_CSV, index=False, float_format="%.6g")
print(amps.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"level {qc['level']:.3f}; kept {int(qc['keep'].sum())} of {len(amps)}; SHA-256 {sha256_of(AMP_CSV)[:16]}...")
kept = stations[qc["keep"]].reset_index(drop=True)
WEIGHTS = Path(RESULT_DIR) / "M53_weights.dat"
write_weights(WEIGHTS, kept)
print(WEIGHTS.read_text())

**Processing as MTUQ does it.** One `ProcessData` object treats records and Green's functions alike: the causal
band-pass `BAND_HZ`; the window of the weights file; and MTUQ's **distance scaling** (glossary
`distance_scaling`), which multiplies every trace by $(r / 100\ \mathrm{km})^{0.5}$ so that near stations do not
outweigh far ones only by being near (`apply_scaling=True`, MTUQ's setting for surface-wave windows). The
Green's functions get `MAX_SHIFT_S` of extra samples at both ends, room for the time shifts of Part 6
(`apply_padding`). A second object without that padding prepares the Green's functions for your least squares,
which use no shifts. The records are given `ORIGIN_USED` as their origin.

In [ ]:
data = mtuq.read(str(REC_DIR / "*.sac"), format="sac", event_id=EVENT_FOLDER,
                 station_id_list=[f"{name}." for name in kept["station"]], tags=["units:m", "type:displacement"])
data.sort_by_distance()
ORIGINS = [Origin({"time": ORIGIN_USED, "latitude": EV_LAT, "longitude": EV_LON, "depth_in_m": 1000.0 * depth,
                   "id": f"{EVENT_FOLDER}_{depth}km"}) for depth in DEPTHS_KM]
for stream in data:
    stream.origin = ORIGINS[DEPTHS_KM.index(REFERENCE_DEPTH_KM)]
settings = dict(filter_type="Bandpass", freq_min=BAND_HZ[0], freq_max=BAND_HZ[1], pick_type="user_supplied",
                window_type="surface_wave", window_length=WINDOW_S, capuaf_file=str(WEIGHTS), apply_scaling=True,
                apply_weights=True)
process = ProcessData(apply_padding=True, time_shift_min=-MAX_SHIFT_S, time_shift_max=MAX_SHIFT_S, **settings)
process_lsq = ProcessData(apply_padding=False, **settings)
data_p = data.map(process)
greens = db.get_greens_tensors(data.get_stations(), ORIGINS)
greens_p = greens.map(process)
greens_lsq = greens.map(process_lsq)
DT_DATA = data_p[0][0].stats.delta
print(f"{len(data_p)} stations, {sum(len(s) for s in data_p)} traces of {data_p[0][0].stats.npts} samples at "
      f"{DT_DATA:g} s; {len(greens_p)} Green's tensors ({len(DEPTHS_KM)} depths)")

**Your least squares at every depth.** For each trial depth, the six elementary seismograms of every trace are
MTUQ's synthetics of the six unit tensors (Part 0) from the unpadded Green's functions; your `invert_mt` finds the
best trace-free tensor (the database holds no change of volume that the records could constrain, so the
deviatoric constraint of Part 1). All samples get the same sigma (1): the fit is MTUQ's L2 norm without shifts,
and `cov` is not used. For each depth: the L2 misfit as a fraction of the records' own L2 sum, your n_eff (task 2)
and your summary (task 4).

In [ ]:
def unit_synthetics(tensors):
    """Elementary seismograms (n samples x 6) of the processed traces, in the order of data_p."""
    columns = []
    for j in range(6):
        unit = np.zeros(6)
        unit[j] = 1.0
        streams = tensors.get_synthetics(MomentTensor(unit), components=data_p.get_components(),
                                         stats=data_p.get_stats(), mode="map")
        columns.append(np.concatenate([tr.data for st_ in streams for tr in st_]))
    return np.column_stack(columns)


trace_ids = [f"{s.station.network}.{s.station.station}.{tr.stats.channel[-1]}" for s in data_p for tr in s]
obs_rows = np.array([tr.data for s in data_p for tr in s])
lsq_rows, lsq_terms = [], {}
for depth, origin in zip(DEPTHS_KM, ORIGINS):
    gmat = unit_synthetics(greens_lsq.select(origin))
    m_hat, _ = invert_mt(gmat, obs_rows.reshape(-1), 1.0, deviatoric=True)
    syn_rows = (gmat @ m_hat).reshape(obs_rows.shape)
    budget = misfit_budget(obs_rows, syn_rows, DT_DATA, "L2")
    info = tensor_summary(m_hat)
    lsq_terms[depth] = budget["per_trace"]
    lsq_rows.append({"depth_km": depth, "misfit": budget["total"] / (DT_DATA * np.sum(obs_rows ** 2)),
                     "n_eff": budget["n_eff"], "mw": magnitude_moment(m0_nm=info["m0_nm"]),
                     "gamma_deg": info["gamma_deg"], "delta_deg": info["delta_deg"],
                     **{f"m{k}": v for k, v in zip(("rr", "tt", "pp", "rt", "rp", "tp"), m_hat)}})
lsq = pd.DataFrame(lsq_rows)
LSQ_CSV = Path(RESULT_DIR) / "M53_lsq.csv"
lsq.to_csv(LSQ_CSV, index=False, float_format="%.10g")
best_row = lsq.loc[lsq["misfit"].idxmin()]
LSQ_DEPTH = int(best_row["depth_km"])
lsq_m6 = best_row[["mrr", "mtt", "mpp", "mrt", "mrp", "mtp"]].to_numpy(dtype=float)     # tensor of LSQ_DEPTH
LSQ_TRACES = Path(RESULT_DIR) / "M53_lsq_traces.csv"
pd.DataFrame({"trace": trace_ids, "term": lsq_terms[LSQ_DEPTH]}).to_csv(LSQ_TRACES, index=False,
                                                                        float_format="%.10g")
print(lsq[["depth_km", "misfit", "n_eff", "mw", "gamma_deg", "delta_deg"]].round(3).to_string(index=False))
print("SHA-256 of", short_path(LSQ_CSV), sha256_of(LSQ_CSV)[:16], "...")

Write your report from the printed rows and your files (the planes and the Kagan angle from your own functions;
`lsq_m6` is the tensor of the best depth).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("M53-T5: write your code here")

In [ ]:
st.check("M53-T5", func=amplitude_check, report=sangju_r2, amplitudes=AMP_CSV, lsq=LSQ_CSV, lsq_traces=LSQ_TRACES,
         summary=tensor_summary, kagan=kagan_angle)

Your report, and how your least-squares tensor relates to the other mechanisms (the depth rows above give the
rest).

In [ ]:
print({k: (np.round(v, 3).tolist() if isinstance(v, (list, np.ndarray)) and k == "lsq_planes" else v)
       for k, v in sangju_r2.items() if not k.endswith("sha256")})
best_info = tensor_summary(lsq_m6)
print(f"least squares at {LSQ_DEPTH} km: Mw {magnitude_moment(m0_nm=best_info['m0_nm']):.2f} "
      f"(KIGAM {KIGAM_MW}), gamma {best_info['gamma_deg']:.1f}, delta {best_info['delta_deg']:.1f}; "
      f"Kagan to KIGAM plane A {sangju_r2['kagan_lsq_kigam']:.1f} deg")

**Explain (3-5 sentences):** Use the printed numbers. Which station did your amplitude check leave out (if any),
by how many powers of ten, and does Part 2 suggest that it would have mattered? How far is your least-squares
tensor from a double couple (gamma) and from KIGAM's mechanism, and how clearly do the misfits pick a depth? Name
one thing that this least-squares fit cannot do that MTUQ's search in Part 6 will. Was your prediction right?

## Part 6 · MTUQ: a double-couple grid search, run twice, then with one change (about 90 min, then task 6)

MTUQ's **grid search** (glossary `grid_search`) tries every double couple of a regular grid (Tape & Tape 2015:
`GRID_POINTS` values each of strike, of dip, through its cosine, and of rake between -90 and 90 degrees, which
with the other plane covers every double couple) at every magnitude of `MAGNITUDES` and every depth of
`DEPTHS_KM`, and computes the misfit of each with the norm you choose (`"L2"` or `"hybrid"`, the two it computes
as named; Part 2), after the best time shift of each station within `MAX_SHIFT_S` (one shift for its three
components, `time_shift_groups=["ZRT"]`). It runs serially here, on
one core, without MPI. Its misfits are divided by the records' own norm (`normalize=True`), so 1 means "no better
than zero synthetics".

Three numbers decide what the best mechanism is worth:
- n_eff and the largest share of its misfit (Part 2): how many traces write the answer;
- the spread: the largest Kagan angle between the best mechanism and the **acceptable mechanisms** (glossary
  `acceptable_mechanism`), those whose misfit lies within `ACCEPT_TOL` (2 %) of the best;
- the Kagan angle to your least squares of Part 5, an independent route through the same records.

A **resolution verdict** (glossary `resolution_verdict`) turns them into one of three words by rules fixed before
the run (`RULES`): "not resolved" when the answer rests on too few stations or traces, "resolved" when it rests on
many and is tight and agrees with the least squares, "provisional" in between. Deciding the rules before looking
at the result is what keeps them honest.

### M53-T6 · MTUQ for the Sangju mainshock, three runs and a verdict · core · about 120 min

**Goal:** write `resolution_verdict(n_stations, n_eff, largest_share, spread_deg, kagan_lsq_deg, rules)`; run
MTUQ's grid search three times (`NORM`, `NORM` again, `CHANGED_NORM`); and report in a dictionary `sangju_r3`.

**Given:** Part 6; the cells below run MTUQ and write `results/M53_mtuq_<run>/misfit.npy` (one misfit per grid
point), `results/M53_mtuq_grid.npz` (the grid's axes) and `results/M53_mtuq_first/traces.csv`. Not allowed in
`resolution_verdict`: MTUQ and the course package.

**Contract (the function):**
- Inputs: the number of stations used, n_eff and the largest share of the best fit's misfit, the spread and the
  Kagan angle to the least squares (degrees), and `rules`, a dictionary with the keys of `RULES`. Do not change
  them.
- "not resolved" when `n_stations < rules["min_stations"]`, or `n_eff < rules["min_n_eff"]`, or
  `largest_share > rules["max_share"]`;
- else "resolved" when `n_eff >= rules["good_n_eff"]` and `largest_share <= rules["good_share"]` and
  `spread_deg <= rules["max_spread_deg"]` and `kagan_lsq_deg <= rules["max_kagan_lsq_deg"]`;
- else "provisional".
- Output: a tuple `(verdict, reasons)`: the word, and a list of the names of the `rules` keys that the inputs
  failed (for "not resolved" the first three tests, for "provisional" the four of "resolved"; empty for
  "resolved"), in the order of this list.
- Example (invented): 7 stations, n_eff 8.0, largest share 0.25, spread 40, Kagan 12 with `RULES` give
  `("provisional", ["max_spread_deg"])`.

**Contract (the report):** `sangju_r3 = {"runs": {"first": ..., "again": ..., "changed": ...}, "norms": {"first":
..., "again": ..., "changed": ...}, "grid_sha256": ..., "best": {"first": [...], "changed": [...]}, "n_stations":
..., "n_eff": ..., "largest_share": ..., "spread_deg": ..., "kagan_kigam": ..., "kagan_skhash": ..., "kagan_lsq":
..., "verdict": ..., "reasons": [...]}`: the SHA-256 of each run's `misfit.npy` and of the grid file; the norm of
each run, as its folder's `norm.txt` names it; each run's best `[depth_km, mw, strike, dip, rake]`; and for the
first run the numbers of the verdict, its Kagan angles to KIGAM's plane A, to your SKHASH mechanism of M51 and to
the first plane of your least squares, and your verdict with its reasons.

**Done when** `st.check("M53-T6", func=resolution_verdict, report=sangju_r3, ...)` (the cell below) passes: 200
made-up sets of numbers around the rules' edges; then your files: the check recomputes the fingerprints, reads the
norm of each run from its folder (it refuses a norm that MTUQ does not compute as named), checks that the rerun is
identical and the changed run is not, each best mechanism from its misfit file, n_eff and the shares from
your trace file, the spread and the three Kagan angles with your `kagan_angle`, and the verdict with your
function and the course's rules. It prints no value of the Sangju data.

**Stuck?** `st.hint("M53-T6", 1)`

In [ ]:
# Before you code: with the stations your amplitude check kept and the hybrid norm, will MTUQ's best double couple
# lie closer to KIGAM's than your least squares did? Will a second run give identical files? With the L2 norm, will
# the best mechanism or the depth move? Which verdict do you expect, and which rule do you expect to decide it?
st.predict("M53-T6", """

""")

In [ ]:
def resolution_verdict(n_stations, n_eff, largest_share, spread_deg, kagan_lsq_deg, rules):
    """'resolved', 'provisional' or 'not resolved' by the rules of Part 6, with the names of the failed rules.

    returns : (verdict, reasons); reasons = names of the rules keys failed, in the order of the contract
    """
    # Step 1: the three tests of "not resolved", in order; if any fails, that is the verdict.
    # Step 2: otherwise the four tests of "resolved", in order; all passed -> "resolved", else "provisional".
    # YOUR CODE HERE
    raise NotImplementedError("M53-T6: write your code here")

**The runs.** `mtuq_run` runs the grid search with one norm and saves its misfits (axes: magnitude, strike, rake,
dip, depth, in the order of `results/M53_mtuq_grid.npz`) and the norm's name (`norm.txt`). It refuses a norm that
MTUQ's grid search does not compute as named (Part 2). Run it as it is: first, then again unchanged, then with
`CHANGED_NORM`. Each run takes seconds to a minute on one core (the first is the slowest: MTUQ compiles its misfit
code with numba on first use).

In [ ]:
GRID = DoubleCoupleGridRegular(npts_per_axis=GRID_POINTS, magnitudes=MAGNITUDES)
MTUQ_NORMS = ("L2", "hybrid")   # the norms that MTUQ's grid search computes under their own names (Part 2)


def mtuq_run(norm, label):
    """MTUQ's grid search with one norm; saves results/M53_mtuq_<label>/misfit.npy and norm.txt, returns the misfits."""
    if norm not in MTUQ_NORMS:
        raise ValueError(f"norm {norm!r}: MTUQ's grid search computes only {MTUQ_NORMS} under their own names (asked "
                         "for 'L1', it computes the L2 sum; Part 2). Set NORM and CHANGED_NORM to one of them.")
    misfit = Misfit(norm=norm, time_shift_min=-MAX_SHIFT_S, time_shift_max=MAX_SHIFT_S,
                    time_shift_groups=["ZRT"], verbose=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = grid_search(data_p, greens_p, misfit, ORIGINS, GRID, msg_interval=0, timed=False, verbose=0)
    values = result.isel(v=0, w=0).transpose("rho", "kappa", "sigma", "h", "origin_idx").values
    folder = Path(RESULT_DIR) / f"M53_mtuq_{label}"
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / "misfit.npy", np.ascontiguousarray(values, dtype=np.float64))
    (folder / "norm.txt").write_text(norm + "\n", encoding="utf-8")
    if label == "first":
        np.savez(Path(RESULT_DIR) / "M53_mtuq_grid.npz", axes=np.array(["mw", "strike", "rake", "dip", "depth_km"]),
                 mw=np.array([to_Mw(v) for v in result["rho"].values]), strike=result["kappa"].values,
                 rake=result["sigma"].values, dip=np.degrees(np.arccos(result["h"].values)),
                 depth_km=np.array(DEPTHS_KM, dtype=float))
    return values


def best_of(values):
    """[depth_km, mw, strike, dip, rake] of the smallest misfit, and that misfit."""
    axes = np.load(Path(RESULT_DIR) / "M53_mtuq_grid.npz")
    i, j, k, l, d = np.unravel_index(np.argmin(values), values.shape)
    return [float(axes["depth_km"][d]), round(float(axes["mw"][i]), 2), float(axes["strike"][j]),
            float(axes["dip"][l]), float(axes["rake"][k])], float(values[i, j, k, l, d])


runs = {}
for label, norm in (("first", NORM), ("again", NORM), ("changed", CHANGED_NORM)):
    start = UTCDateTime()
    runs[label] = mtuq_run(norm, label)
    print(f"{label:8s} norm {norm:7s} {runs[label].size} misfits in {UTCDateTime() - start:5.1f} s, SHA-256 "
          f"{sha256_of(Path(RESULT_DIR) / f'M53_mtuq_{label}' / 'misfit.npy')[:16]}...")

**The budget of the first run's best mechanism.** MTUQ's `collect_synthetics` gives, for the best source, each
trace's synthetic with the shift MTUQ chose and MTUQ's own misfit of that trace (here without the division by the
records' norm). Your `misfit_budget` on the record and the shifted synthetic must give the same number; the shares
and n_eff then come from your terms. The trace file is `results/M53_mtuq_first/traces.csv`.

In [ ]:
best_first, best_misfit = best_of(runs["first"])
depth_index = DEPTHS_KM.index(int(best_first[0]))
best_source = MomentTensor(to_mij(to_rho(best_first[1]), 0.0, 0.0, best_first[2], best_first[4],
                                  np.cos(np.radians(best_first[3]))))
plain = Misfit(norm=NORM, time_shift_min=-MAX_SHIFT_S, time_shift_max=MAX_SHIFT_S, time_shift_groups=["ZRT"],
               normalize=False, verbose=0)
synthetics = plain.collect_synthetics(data_p, greens_p.select(ORIGINS[depth_index]), best_source)
trace_rows = []
for obs_stream, syn_stream in zip(data_p, synthetics):
    for obs_trace in obs_stream:
        syn_trace = syn_stream.select(component=obs_trace.stats.channel[-1])[0]
        cut = syn_trace.data[syn_trace.attrs.idx_start:syn_trace.attrs.idx_stop]
        term = misfit_budget(obs_trace.data[None, :], cut[None, :], DT_DATA, NORM)["total"]
        trace_rows.append({"station": f"{obs_stream.station.network}.{obs_stream.station.station}",
                           "component": obs_trace.stats.channel[-1], "shift_s": float(syn_trace.attrs.time_shift),
                           "mtuq_misfit": float(syn_trace.attrs.misfit), "your_term": float(term)})
traces = pd.DataFrame(trace_rows)
traces["share"] = traces["your_term"] / traces["your_term"].sum()
TRACES_CSV = Path(RESULT_DIR) / "M53_mtuq_first" / "traces.csv"
traces.to_csv(TRACES_CSV, index=False, float_format="%.10g")
print(traces.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(f"largest |your term / MTUQ's - 1|: {np.max(np.abs(traces['your_term'] / traces['mtuq_misfit'] - 1)):.2e}")
print("share per station:", traces.groupby("station")["share"].sum().round(3).to_dict())

The records (black) and the best synthetics with MTUQ's shifts (red), in the windows that MTUQ compared (both
distance-scaled; one scale per row).

In [ ]:
fig, axes_ = plt.subplots(len(data_p), 3, figsize=(10, 1.3 * len(data_p)), sharex=True, squeeze=False)
for r, (obs_stream, syn_stream) in enumerate(zip(data_p, synthetics)):
    top = max(np.abs(tr.data).max() for tr in obs_stream)
    for c, comp in enumerate("ZRT"):
        ax = axes_[r, c]
        found_ = obs_stream.select(component=comp)
        if found_:
            obs_trace = found_[0]
            syn_trace = syn_stream.select(component=comp)[0]
            t = np.arange(obs_trace.stats.npts) * DT_DATA
            ax.plot(t, obs_trace.data / top, color="black", lw=0.8)
            ax.plot(t, syn_trace.data[syn_trace.attrs.idx_start:syn_trace.attrs.idx_stop] / top, color="red", lw=0.8)
        ax.set_ylim(-1.1, 1.1)
        ax.set_yticks([])
        if r == 0:
            ax.set_title(comp)
        if c == 0:
            ax.set_ylabel(obs_stream.station.station, rotation=0, ha="right", va="center")
for ax in axes_[-1]:
    ax.set_xlabel("Time in window (s)")
fig.savefig(Path(FIGURE_DIR) / "M53_fits.png", dpi=150, bbox_inches="tight")
plt.show()

**The verdict's numbers.** The spread over the acceptable mechanisms of the first run (all depths and
magnitudes), the comparisons with KIGAM, with your SKHASH mechanism of M51 (task M51-T5) and with your least
squares, and the best misfit at each depth for the first and the changed run.

In [ ]:
if not Path(SKHASH_CSV).is_file():
    raise FileNotFoundError(f"{SKHASH_CSV} is missing. This part compares MTUQ with your SKHASH mechanism: finish "
                            "task M51-T5 of module M51 first (it copies the run to results/M51_skhash_first).")
skhash = pd.read_csv(SKHASH_CSV, dtype={"event_id": str})
skhash_row = skhash.loc[skhash["event_id"] == EVENT_FOLDER].iloc[0]
SKHASH_MECH = (float(skhash_row["strike"]), float(skhash_row["dip"]), float(skhash_row["rake"]))
grid_axes = np.load(Path(RESULT_DIR) / "M53_mtuq_grid.npz")
accepted = np.argwhere(runs["first"] <= (1.0 + ACCEPT_TOL) * best_misfit)
best_mech = tuple(best_first[2:])
angles = [kagan_angle(best_mech, (grid_axes["strike"][j], grid_axes["dip"][l], grid_axes["rake"][k]))
          for _, j, k, l, _ in accepted]
spread = float(max(angles))
n_eff_first = float(1.0 / np.sum(traces["share"] ** 2))
largest_share = float(traces["share"].max())
n_used = int(traces["station"].nunique())
kagan_kigam = kagan_angle(best_mech, KIGAM_PLANE_A)
kagan_skhash = kagan_angle(best_mech, SKHASH_MECH)
kagan_lsq = kagan_angle(best_mech, tensor_summary(lsq_m6)["planes"][0])
print(f"first run: best {best_first} (depth, Mw, strike, dip, rake), misfit {best_misfit:.4f}")
print(f"{len(accepted)} acceptable mechanisms at depths {sorted(set(DEPTHS_KM[d] for *_, d in accepted))} km and "
      f"Mw {sorted(set(round(float(grid_axes['mw'][i]), 2) for i, *_ in accepted))}; spread {spread:.1f} deg")
print(f"n_stations {n_used}, n_eff {n_eff_first:.2f} of {len(traces)} traces, largest share {largest_share:.3f}")
print(f"Kagan angles: to KIGAM plane A {kagan_kigam:.1f}, to SKHASH {SKHASH_MECH} {kagan_skhash:.1f}, to your "
      f"least squares {kagan_lsq:.1f} deg")
best_changed, _ = best_of(runs["changed"])
print(f"changed run ({CHANGED_NORM}): best {best_changed}, Kagan to the first run's "
      f"{kagan_angle(best_mech, tuple(best_changed[2:])):.1f} deg")
by_depth = pd.DataFrame({"depth_km": DEPTHS_KM,
                         f"first ({NORM})": runs["first"].min(axis=(0, 1, 2, 3)),
                         f"changed ({CHANGED_NORM})": runs["changed"].min(axis=(0, 1, 2, 3))})
print(by_depth.round(4).to_string(index=False))
verdict, reasons = resolution_verdict(n_used, n_eff_first, largest_share, spread, kagan_lsq, RULES)
print("your verdict:", verdict, reasons)

Write your report from the printed numbers and your files.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("M53-T6: write your code here")

In [ ]:
st.check("M53-T6", func=resolution_verdict, report=sangju_r3, results=RESULT_DIR, skhash=SKHASH_CSV,
         lsq=LSQ_CSV, summary=tensor_summary, kagan=kagan_angle)

The two norms side by side: each run's best misfit at every depth, and where the best mechanisms of the two runs
lie.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
for column, style in zip(by_depth.columns[1:], ("o-", "s--")):
    ax.plot(by_depth["depth_km"], by_depth[column] / by_depth[column].min(), style, label=column)
ax.set_xlabel("Depth (km)")
ax.set_ylabel("Best misfit / smallest")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
fig.savefig(Path(FIGURE_DIR) / "M53_depth.png", dpi=150, bbox_inches="tight")
plt.show()
print("first:", best_first, " changed:", best_changed)

**Explain (3-5 sentences):** Use the printed numbers. Were the first run and the rerun identical, and why must they
be? What did the changed norm do to the best mechanism, the depth and the depth curve, and which traces does L2
weigh more than hybrid? Look at the shifts in your trace table: how many sit at or next to `MAX_SHIFT_S`, with
which sign, and what could that say about the model's travel times and about the depth? The rules count shares per
trace: which station holds the largest share when its three traces are added, and would a rule per station change
your verdict? Which rule decided your verdict, and how do MTUQ, KIGAM, SKHASH and your least squares compare (which
of these comparisons are independent of one another)?

## Summary

- A point source makes records that are linear in its six moment-tensor elements; weighted least squares finds
  them and its covariance describes their errors when the noise is what the weights say (Part 1).
- Which traces write an answer is a property of the norm and the data, not of the fit's total: the shares and
  n_eff show it, and an amplitude check before the fit removes records whose size is wrong (Parts 2, 5).
- Time shifts let a 1-D model fit records whose travel times it gets a little wrong. A looser bound helps the wrong
  answer too: on the made-up network the opposite mechanism's fit rose most between bounds of 1 and 2 s, well
  before half the period, and stayed far below the made-up fault's, because a band-passed pulse is only partly
  like its own negative (Part 3).
- The lune and the double-couple planes describe a tensor; the Kagan angle compares mechanisms (Part 4).
- MTUQ with your Green's function database, your windows and your amplitude decision gives a mechanism, a depth
  and a magnitude; it is worth what its budget, its spread and an independent route say, decided by rules fixed
  before the run (Part 6).

**Credits.** Instrument responses: StationMetadata by Byeong-Seok Ahn (PKNU; Ahn et al. 2024). Records: KMA
(through STP, module M03). Mechanism and magnitude: KIGAM (2021), report KEMS-2019-062. Green's functions: PyFK
by Xi et al. (https://github.com/ziyixi/pyfk), the FK method of Zhu and Rivera (2002); velocity model Kim and Kim
(1983). Grid search: MTUQ (Thurin et al. 2025; Silwal and Tape 2016; Alvizuri et al. 2018). Module by Minseong Seo
(SGTL, Seoul National University).